# 6. Mouse-Only Pseudospace Reconstruction

Reconstructs kidney-nephron **pseudospace** from segmented Visium tubules for the **4 mouse samples**
(`Ctrl1A2`, `Ctrl1A4`, `IR2A2`, `IR2A4`) in native mouse gene space (no ortholog mapping), then
compares healthy vs AKI trajectories.

**Workflow**
0. Config — every path and parameter.
1. Harmony **pass 1** (all tubules, intersection HVGs).
2. Classic marker-based cell typing on a **two-pass Harmony** design: coarse dotplot typing → remove
   all non-nephron-epithelial cells → **re-integrate** (pass 2) → fine dotplot typing → Scanpy DPT on
   the pass-2 embedding. Includes injury/failed-repair diagnostics and a theta sweep.
3. Marker heatmaps.
4. Healthy vs AKI — nested level/shape GAMs with honest 2v2 sample permutation + leave-one-specimen-out
   robustness.
5. Three-axis concordance (pseudospace vs marker axis vs distance-to-glomerulus).

All outputs are written under `results/mouse_only/` (the older `data/mouse_only/` tree is left
untouched). Requires the `pseudospace-agproject` kernel (`rpy2` + R `harmony`, `python-igraph`,
`leidenalg`).

## Section 0 — Config

Every path and parameter used below. All later sections reference these variables only.

In [ ]:
# Development: auto-reload the local `pseudospace` package when its modules are edited.
# The notebook runs from the repository root, so `import pseudospace...` resolves directly
# (no sys.path manipulation needed).
%load_ext autoreload
%autoreload 2

In [ ]:
from __future__ import annotations

import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings('ignore', category=FutureWarning)

# ---------------------------------------------------------------------------
# Paths -- inputs stay under data/, ALL outputs go under results/mouse_only_v4/.
# data/mouse_only_v4/ is left untouched (a prior stale run is preserved there).
# ---------------------------------------------------------------------------
PROJECT_DIR = Path.cwd().resolve()
DATA_DIR = PROJECT_DIR / 'data'

TUBULE_BY_GENE_DIR = DATA_DIR / 'tubule_by_gene'                              # input
PATHWAY_LIBRARY_DIR = DATA_DIR / 'mouse_vs_human' / 'pathway_gene_sets'       # input (read-only)

RESULTS_DIR = PROJECT_DIR / 'results' / 'mouse_only_v4'
HARMONY_OUTPUT_PATH = RESULTS_DIR / 'all_mouse_tubules_harmony_pass1.h5ad'    # pass-1, all cells
PASS2_HARMONY_OUTPUT_PATH = RESULTS_DIR / 'all_mouse_tubules_harmony.h5ad'    # pass-2, tubule-only
DPT_OUTPUT_PATH = RESULTS_DIR / 'all_mouse_tubules_scanpy_dpt.h5ad'
LEIDEN_EXPLORATION_DIR = RESULTS_DIR / 'leiden_exploration'
HEATMAP_OUTPUT_DIR = RESULTS_DIR / 'heatmaps'
HEALTHY_VS_AKI_OUTPUT_DIR = RESULTS_DIR / 'healthy_vs_aki'
INJURY_DIAG_DIR = RESULTS_DIR / 'injury_diagnostics'
CONCORDANCE_DIR = RESULTS_DIR / 'concordance'

for directory in [RESULTS_DIR, LEIDEN_EXPLORATION_DIR, HEATMAP_OUTPUT_DIR,
                  HEALTHY_VS_AKI_OUTPUT_DIR, INJURY_DIAG_DIR, CONCORDANCE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

R_LIB_DIR = PROJECT_DIR / 'r_libs'
R_LIB_DIR.mkdir(exist_ok=True)
os.environ.setdefault('R_LIBS_USER', str(R_LIB_DIR))

CACHE_ROOT = Path('/tmp/pseudospace_mouse_only_v4_cache')
CACHE_ROOT.mkdir(exist_ok=True)
(CACHE_ROOT / 'matplotlib').mkdir(exist_ok=True)
(CACHE_ROOT / 'numba').mkdir(exist_ok=True)
(CACHE_ROOT / 'xdg').mkdir(exist_ok=True)
os.environ.setdefault('NUMBA_CACHE_DIR', str(CACHE_ROOT / 'numba'))
os.environ.setdefault('MPLCONFIGDIR', str(CACHE_ROOT / 'matplotlib'))
os.environ.setdefault('XDG_CACHE_HOME', str(CACHE_ROOT / 'xdg'))

# ---------------------------------------------------------------------------
# Sample scope
# ---------------------------------------------------------------------------
MOUSE_SAMPLES = ['Ctrl1A2', 'Ctrl1A4', 'IR2A2', 'IR2A4']
BATCH_KEY = 'sample'
RANDOM_STATE = 0

# ---------------------------------------------------------------------------
# Harmony / HVG params. BOTH Harmony passes use INTERSECTION HVGs
# (condition-invariant features; avoids feeding condition-associated genes to Harmony).
# ---------------------------------------------------------------------------
HARMONY_HVG_MIN_MEAN = 0.0125
HARMONY_HVG_MAX_MEAN = 3
HARMONY_HVG_MIN_DISP = 0.5

HARMONY_N_PCS = 50
# HARMONY_THETA_GRID = [1, 2, 4, 8]   # swept in Section 2d; inspect mixing vs bio-conservation
HARMONY_THETA = 6                    # <- set from the theta sweep; used for BOTH passes
HARMONY_LAMBDA = 1
HARMONY_MAX_ITER = 30
HARMONY_TAU = 0
HARMONY_PCA_N_COMPS = 50
HARMONY_NEIGHBORS_N = 25
HARMONY_UMAP_MIN_DIST = 0.3
HARMONY_UMAP_SPREAD = 1.0

# ---------------------------------------------------------------------------
# Gene filter / clustering / DPT params
# ---------------------------------------------------------------------------
MIN_GENE_TUBULE_FRACTION = 0.05
MIN_GENE_TOTAL_COUNTS = 20
NORMALIZE_TARGET_SUM = 1e4
N_HVGS = 2000

N_NEIGHBORS = 30
N_PCS_FALLBACK = 30
N_DIFFMAP_COMPONENTS_TO_TEST = 8

COARSE_RESOLUTION = 0.5                 # pass-1 coarse clustering (identify non-tubule)
RESOLUTIONS_TO_EXPLORE = [0.3, 0.4, 0.5, 0.6, 0.8, 1.0, 1.2]   # pass-2 fine clustering
SELECTED_RESOLUTION = None              # set at the fine manual checkpoint
EXCLUDE_INJURY_FROM_DPT = False         # decided after the injury diagnostics (Section 2c)

# ---------------------------------------------------------------------------
# Heatmap params
# ---------------------------------------------------------------------------
TRIM_FRACTION = 0.05
N_BINS = 120
SMOOTH_SIGMA = 2.5
Z_CLIP = 2

# ---------------------------------------------------------------------------
# Marker panels (mouse symbols). Curated markers are NEVER gated out by the
# expression filter (whitelist below) and drive both the dotplot cell typing and
# the whitelist. Genes absent from the mouse gene space (Nphs1, Akr1b1, Fxyd4,
# Trpm6, Nos1, Ptgs2) are intentionally omitted.
# ---------------------------------------------------------------------------
# NON_TUBULE_MARKERS = {
#     'Glomerulus':    ['Nphs2', 'Wt1', 'Podxl', 'Synpo', 'Ptgds'],
#     'Vessel':        ['Pecam1', 'Cdh5', 'Kdr', 'Emcn', 'Flt1', 'Egfl7'],
#     'Stroma':        ['Col1a1', 'Col3a1', 'Col1a2', 'Dcn', 'Pdgfrb', 'Mgp'],
#     'SmoothMuscle':  ['Acta2', 'Myh11', 'Tagln', 'Rgs5', 'Pdgfra'],
#     'Immune':        ['Ptprc', 'C1qa', 'C1qb', 'Cd52', 'Lyz2', 'Cd74'],
# }
# # Broad tubule families -- the coarse dotplot + the label vocabulary for broad_tubule_marker_call.
# BROAD_TUBULE_MARKERS = {
#     'PT':     ['Slc34a1', 'Lrp2', 'Slc5a12', 'Slc22a6'],
#     'TL':     ['Slc14a2', 'Bst1', 'Sostdc1', 'Cldn3', 'Aqp1'],
#     'TAL':    ['Umod', 'Slc12a1', 'Cldn10', 'Egf'],
#     'DCT':    ['Slc12a3', 'Pvalb', 'Kcnj10'],          # Pvalb recovered via whitelist; Kcnj10 = DCT1 anchor
#     'CNT_CD': ['Calb1', 'Aqp2', 'Aqp3', 'Slc4a1', 'Slc26a4', 'Atp6v1b1'],
# }
# BROAD_FAMILIES = list(BROAD_TUBULE_MARKERS)
# # Fine nephron-segment dotplot (visual sub-structure; PT S1/S2/S3 etc.).
# SEGMENT_MARKERS = {
#     'PT-S1':  ['Slc5a2', 'Slc5a12', 'Gatm'],
#     'PT-S2':  ['Slc22a6', 'Slc22a8', 'Slc13a3', 'Slc34a1'],
#     'PT-S3':  ['Slc7a13', 'Slc5a8', 'Hao2', 'Pck1', 'Acsm3'],
#     'TL':     ['Aqp1', 'Clcnka', 'Bst1', 'Sostdc1', 'Slc14a2', 'Cldn3'],
#     'TAL':    ['Umod', 'Slc12a1', 'Cldn10', 'Cldn16', 'Egf', 'Kcnj1', 'Casr'],
#     'DCT':    ['Slc12a3', 'Pvalb', 'Kcnj10'],
#     'CNT':    ['Calb1', 'Slc8a1', 'Scnn1g', 'Trpv5'],   # Trpv5 recovered via whitelist
#     'CD_PC':  ['Aqp2', 'Aqp3'],
#     'CD_IC':  ['Atp6v1b1', 'Atp6v1g3', 'Slc4a1', 'Slc26a4'],
# }
# INJURY_MARKERS = ['Vcam1', 'Havcr1', 'Krt20', 'Spp1', 'Sox9', 'Lcn2',
#                   'Krt8', 'Krt19', 'Sprr1a', 'Cd24a', 'Gdf15']
# TOTAL_POSITION_MARKERS = {
#     'early': ['Slc34a1', 'Lrp2', 'Slc5a2', 'Slc5a12'],
#     'late':  ['Aqp2', 'Aqp3', 'Slc4a1', 'Slc26a4', 'Atp6v1b1'],
# }

# Caleb recurated

NON_TUBULE_MARKERS = {
    'Glomerulus':    ['Nphs2', 'Podxl', 'Nphs1', 'Mafb'],
    'Vessel':        ['Pecam1', 'Cdh5', 'Kdr', 'Emcn', 'Flt1', 'Egfl7'],
    'Stroma':        ['Col1a1', 'Col3a1', 'Col1a2', 'Dcn', 'Pdgfrb', 'Mgp'],
    'SmoothMuscle':  ['Acta2', 'Myh11', 'Tagln', 'Rgs5', 'Pdgfra'],
    'Immune':        ['Ptprc', 'C1qa', 'C1qb', 'Cd52', 'Lyz2', 'Cd74'],
}
# Broad tubule families -- the coarse dotplot + the label vocabulary for broad_tubule_marker_call.
BROAD_TUBULE_MARKERS = {
    'Glomerulus':    ['Nphs2', 'Podxl', 'Nphs1', 'Mafb'],
    'PT':     ['Slc3a1', 'Slc6a14', 'Slc7a8', 'Slc22a5', 'Slc9a3', 'Slc4a4', 'Slc5a2'],
    'DTL':    ['Aqp1'],                                        # Canonical unique DTL marker (Slc20a1 in chart is ubiquitous)
    'ATL':    ['Clcnka'],                                      # Strictly unique to ATL
    'TAL':    ['Slc12a1'],                                     # Strictly unique to TAL
    'DCT':    ['Slc12a3', 'Calb1'],                            # Strictly unique to DCT (DCL)
    'CNT_CD': ['Slc8a1', 'Aqp2', 'Aqp3', 'Slc4a1', 'Slc26a4', 'Atp6v1b1'], # Strictly unique to CNT/CD
}

BROAD_FAMILIES = list(BROAD_TUBULE_MARKERS)

BROAD_FAMILIES = list(BROAD_TUBULE_MARKERS)
# Fine nephron-segment dotplot (visual sub-structure; PT S1/S2/S3 etc.).
SEGMENT_MARKERS = {
    'Glomerulus':    ['Nphs2', 'Podxl', 'Nphs1', 'Mafb'],
    'PT-S1':  ['Slc5a2'],           # Peak expression in S1 (spills into S2)
    'PT-S2':  ['Slc1a1', 'Slc22a2'],# Peak expression in S2/S3
    'PT-S3':  ['Slc5a1', 'Slc6a13'],# Strictly unique to S3
    'DTL':    ['Slc20a1'],          # Primary thin limb marker present in chart
    'ATL':    ['Clcnka'],           # Strictly unique to ATL
    'TAL':    ['Slc12a1'],          # Strictly unique to TAL
    'DCT':    ['Slc12a3', 'Calb1'], # Strictly unique to DCT
    'CNT':    ['Slc8a1'],           # Strictly unique to CNT
    'CD':     ['Slc42a3'],          # Primary CD marker in chart (shared with DCL/CNT)
}
INJURY_MARKERS = ['Vcam1', 'Havcr1', 'Krt20', 'Spp1', 'Sox9', 'Lcn2',
                  'Krt8', 'Krt19', 'Sprr1a', 'Cd24a', 'Gdf15']
TOTAL_POSITION_MARKERS = {
    'early': ['Slc5a2', 'Slc6a19', 'Slc4a4'],
    'late':  ['Slc42a3', 'Clcnkb'],
}
CURATED_MARKER_WHITELIST = sorted({
    g
    for panel in (list(NON_TUBULE_MARKERS.values()) + list(BROAD_TUBULE_MARKERS.values())
                  + list(SEGMENT_MARKERS.values()) + [INJURY_MARKERS]
                  + list(TOTAL_POSITION_MARKERS.values()))
    for g in panel
})

print(f'Results directory:          {RESULTS_DIR.relative_to(PROJECT_DIR)}')
print(f'Mouse samples:              {MOUSE_SAMPLES}')
print(f'Pass-1 Harmony output:      {HARMONY_OUTPUT_PATH.relative_to(PROJECT_DIR)}')
print(f'Pass-2 Harmony output:      {PASS2_HARMONY_OUTPUT_PATH.relative_to(PROJECT_DIR)}')
print(f'DPT output:                 {DPT_OUTPUT_PATH.relative_to(PROJECT_DIR)}')
print(f'Curated markers whitelisted: {len(CURATED_MARKER_WHITELIST)}')


## Section 1 — Mouse-Only Harmony Integration

Adapted from `1_segmentation_to_gene_matrix.ipynb` Part 3, with all ortholog machinery removed: no
HCOP table, no `get_hcop_ortholog_map` / `convert_human_to_mouse_symbols` /
`filter_mouse_to_ortholog_space`. Samples stay in native mouse gene symbols throughout, and the two
human samples are never loaded.

In [ ]:
import anndata as ad
import scanpy as sc
import rpy2.robjects as ro
from rpy2.robjects import numpy2ri, pandas2ri
from rpy2.robjects.conversion import localconverter
from rpy2.robjects.packages import importr

ro.r('.libPaths(c("~/R/library", .libPaths()))')

print('rpy2 / R library path configured.')

In [ ]:
# QC helpers moved to pseudospace/io_qc.py. `calculate_qc_metrics` is used
# internally by `load_and_process_mouse_samples` and does not need importing here.
from pseudospace.io_qc import annotate_mito_ribo_mouse_symbols

In [ ]:
from pseudospace.io_qc import (
    sample_name_from_path,
    load_and_process_mouse_samples,
    combine_adatas,
)

In [ ]:
files = sorted(path.name for path in TUBULE_BY_GENE_DIR.glob('*_tubule_by_gene_caleb.h5ad'))
files = [f for f in files if 'HUK' not in f]

if not files:
    raise FileNotFoundError(f'No mouse tubule-by-gene files found in {TUBULE_BY_GENE_DIR}')

found_samples = sorted(sample_name_from_path(Path(f)) for f in files)
if found_samples != sorted(MOUSE_SAMPLES):
    raise FileNotFoundError(
        f'Expected exactly {sorted(MOUSE_SAMPLES)} mouse samples, found {found_samples}. '
        'Check data/tubule_by_gene/ for missing *_tubule_by_gene_caleb.h5ad files.'
    )
print(f'Discovered {len(files)} mouse sample files: {found_samples}')

mouse_adatas = load_and_process_mouse_samples(TUBULE_BY_GENE_DIR, files)
adata_combined = combine_adatas(mouse_adatas)
adata_combined = annotate_mito_ribo_mouse_symbols(adata_combined)

print(f'\nCombined: {adata_combined.n_obs:,} tubules x {adata_combined.n_vars:,} genes (native mouse space)')
print(adata_combined.obs.groupby('condition')['sample'].value_counts())

In [ ]:
print(f'Before cell filtering: {adata_combined.n_obs} tubules x {adata_combined.n_vars} genes')
sc.pp.filter_cells(adata_combined, min_genes=100)
print(f'After cell filtering: {adata_combined.n_obs} tubules x {adata_combined.n_vars} genes')

adata_combined.layers['counts'] = adata_combined.X.copy()

sc.pp.normalize_total(adata_combined, target_sum=NORMALIZE_TARGET_SUM)
sc.pp.log1p(adata_combined)

adata_combined.layers['lognorm'] = adata_combined.X.copy()
print("Normalized; stored counts in layers['counts'] and log-normalized values in layers['lognorm'].")

In [ ]:
from pseudospace.harmony import select_harmony_hvgs_by_condition

In [ ]:
adata_combined = select_harmony_hvgs_by_condition(
    adata_combined,
    group_key='condition',
    groups=('Control', 'IR'),
    mode='intersection',
    # min_groups=HARMONY_HVG_MIN_GROUPS,
    min_mean=HARMONY_HVG_MIN_MEAN,
    max_mean=HARMONY_HVG_MAX_MEAN,
    min_disp=HARMONY_HVG_MIN_DISP,
)

hvg_col = 'highly_variable_for_harmony'
print(f'\nFinal Harmony features: {int(adata_combined.var[hvg_col].sum())}')
print('HVG condition-count distribution:')
print(adata_combined.var['hvg_condition_count'].value_counts().sort_index())

In [ ]:
from pseudospace.harmony import run_harmony_rpy2

In [ ]:
adata_hvg = adata_combined[:, adata_combined.var[hvg_col]].copy()
adata_hvg.X = adata_hvg.layers['lognorm'].copy()

# Compute the (uncorrected) PCA FIRST, then use it for the pre-Harmony UMAP.
# (Previously this cell copied adata_hvg.obsm['X_pca'] before it existed.)
sc.tl.pca(adata_hvg, n_comps=HARMONY_PCA_N_COMPS)

# ---- UMAP BEFORE Harmony (uncorrected PCA) ----
adata_combined.obsm['X_pca'] = adata_hvg.obsm['X_pca'].copy()

sc.pp.neighbors(
    adata_combined,
    use_rep='X_pca',
    n_neighbors=HARMONY_NEIGHBORS_N,
    key_added='pre_harmony',
)
sc.tl.umap(
    adata_combined,
    min_dist=HARMONY_UMAP_MIN_DIST,
    spread=HARMONY_UMAP_SPREAD,
    neighbors_key='pre_harmony',
)
adata_combined.obsm['X_umap_pre_harmony'] = adata_combined.obsm['X_umap'].copy()

sc.pl.umap(
    adata_combined,
    color=['sample', 'condition'],
    ncols=2,
    frameon=False,
    title=['Pre-Harmony: sample', 'Pre-Harmony: condition'],
)
# --------------------------------------------------


In [ ]:

print(f'\nRunning Harmony on {adata_hvg.n_obs} tubules x {adata_hvg.n_vars} selected features')
print(adata_hvg.obs['sample'].value_counts())

# (PCA already computed in the pre-Harmony cell above)
adata_hvg = run_harmony_rpy2(
    adata_hvg,
    batch_key=BATCH_KEY,
    n_pcs=HARMONY_N_PCS,
    theta=HARMONY_THETA,
    lambda_val=HARMONY_LAMBDA,
    max_iter=HARMONY_MAX_ITER,
    tau=HARMONY_TAU,
)

adata_combined.obsm['X_harmony'] = adata_hvg.obsm['X_harmony'].copy()
adata_combined.obsm['X_pca'] = adata_hvg.obsm['X_pca'].copy()

sc.pp.neighbors(adata_combined, use_rep='X_harmony', n_neighbors=HARMONY_NEIGHBORS_N)
sc.tl.umap(adata_combined, min_dist=HARMONY_UMAP_MIN_DIST, spread=HARMONY_UMAP_SPREAD)

sc.pl.umap(adata_combined, color=['sample', 'condition'], ncols=2, frameon=False)

In [ ]:
adata_combined.write(HARMONY_OUTPUT_PATH)

print(f'\nSaved pass-1 harmonized mouse-only AnnData to: {HARMONY_OUTPUT_PATH.relative_to(PROJECT_DIR)}')
print('\n--- Section 1 summary ---')
print(f'Tubules: {adata_combined.n_obs:,}')
print(f'Genes (native mouse space): {adata_combined.n_vars:,}')
print(f'Harmony features (intersection HVG, non-mito/ribo): {int(adata_combined.var["highly_variable_for_harmony"].sum()):,}')
print(f'X_harmony shape: {adata_combined.obsm["X_harmony"].shape}')
print(f'X_pca shape: {adata_combined.obsm["X_pca"].shape}')

## Section 2 — Cell typing (two-pass Harmony) + Scanpy DPT

Classic marker-based cell typing on a **two-pass Harmony** design:
**2a** whitelist-protected gene filter → **2b** coarse Leiden + canonical-marker dotplot → assign a
`cluster→class` dict and drop all non-nephron-epithelial cells → **2c** injury/failed-repair
diagnostics → **2d** re-integrate the tubule subset (pass 2), fine-type by dotplot, and build Scanpy
DPT on the pass-2 embedding. Labels are assigned from an explicit, reviewable dict; identity is
validated against spatial position (Section 5), not the same marker panels.

In [ ]:
import scanpy as sc
import anndata as ad
from scipy import sparse
from scipy.stats import spearmanr
from sklearn.metrics import (
    adjusted_rand_score,
    calinski_harabasz_score,
    davies_bouldin_score,
    normalized_mutual_info_score,
    silhouette_score,
)

try:
    import igraph  # noqa: F401
except ImportError as exc:
    raise ImportError(
        'python-igraph is required for sc.tl.leiden(flavor="igraph") in Section 2. '
        'Install it in this kernel environment before proceeding.'
    ) from exc

adata_all = sc.read_h5ad(HARMONY_OUTPUT_PATH)
print(f'Loaded harmonized mouse-only object: {adata_all.n_obs:,} tubules x {adata_all.n_vars:,} genes')

In [ ]:
if 'counts' in adata_all.layers:
    counts = adata_all.layers['counts'].copy()
else:
    counts = adata_all.X.copy()
    print('No counts layer found; using current X as the raw-count source')
adata_all.layers['counts'] = counts.copy()

counts_matrix = counts if sparse.issparse(counts) else np.asarray(counts)
expressed = np.asarray((counts_matrix > 0).sum(axis=0)).ravel()
total_counts = np.asarray(counts_matrix.sum(axis=0)).ravel()
min_cells = max(1, int(np.ceil(MIN_GENE_TUBULE_FRACTION * adata_all.n_obs)))
gene_keep = (expressed >= min_cells) & (total_counts >= MIN_GENE_TOTAL_COUNTS)
# Never gate out curated markers -- protect the whitelist through the filter.
whitelist_hits = adata_all.var_names.str.upper().isin({g.upper() for g in CURATED_MARKER_WHITELIST})
gene_keep = gene_keep | np.asarray(whitelist_hits)
adata_all.var['n_tubules_expressed'] = expressed
adata_all.var['total_counts'] = total_counts
adata_all.var['passes_expression_count_filter'] = gene_keep
print(
    f'Gene filter keeps {int(gene_keep.sum()):,}/{adata_all.n_vars:,} genes '
    f'(expressed in >= {MIN_GENE_TUBULE_FRACTION:.0%} tubules and total counts >= {MIN_GENE_TOTAL_COUNTS})'
)
adata_all = adata_all[:, gene_keep].copy()

adata_all.X = adata_all.layers['counts'].copy()
adata_all.uns.pop('log1p', None)
sc.pp.normalize_total(adata_all, target_sum=NORMALIZE_TARGET_SUM)
sc.pp.log1p(adata_all)
adata_all.layers['lognorm'] = adata_all.X.copy()

# Snapshot before HVG subsetting -- this is what gets written to disk in Section 2i,
# with the pseudotime / cluster-label obs columns re-attached by obs_name.
adata_all_expression_filtered = adata_all.copy()
print(f'After gene filter + normalization: {adata_all.n_obs:,} tubules x {adata_all.n_vars:,} genes')

In [ ]:
# --- Section 2b: coarse clustering for non-tubule removal ---
import scanpy as sc
import matplotlib.pyplot as plt
from pseudospace.markers import build_gene_lookup, available_markers_by_group, assign_cluster_labels

gene_lookup = build_gene_lookup(adata_all)

# Clustering feature space = HVGs UNION curated markers (curated markers were also
# protected from the expression filter above, so they are never gated out).
sc.pp.highly_variable_genes(adata_all, n_top_genes=min(N_HVGS, adata_all.n_vars), flavor='seurat')
whitelist_present = {gene_lookup[g.upper()] for g in CURATED_MARKER_WHITELIST if g.upper() in gene_lookup}
feature_mask = adata_all.var['highly_variable'].to_numpy(dtype=bool) | adata_all.var_names.isin(whitelist_present)
adata_all.var['selected_for_clustering'] = feature_mask
adata_cluster = adata_all[:, feature_mask].copy()
gene_lookup = build_gene_lookup(adata_cluster)
print(f'Clustering feature space: {int(feature_mask.sum()):,} genes '
      f'({int(adata_all.var["highly_variable"].sum()):,} HVGs + curated markers)')

# Coarse Leiden on the pass-1 Harmony graph.
sc.pp.neighbors(adata_cluster, n_neighbors=N_NEIGHBORS, use_rep='X_harmony', random_state=RANDOM_STATE)
sc.tl.leiden(adata_cluster, resolution=COARSE_RESOLUTION, key_added='leiden_coarse',
             flavor='igraph', n_iterations=2, directed=False, random_state=RANDOM_STATE)
print(f'Coarse Leiden (res={COARSE_RESOLUTION}): {adata_cluster.obs["leiden_coarse"].nunique()} clusters')
sc.tl.rank_genes_groups(adata_cluster, 'leiden_coarse', method='wilcoxon', pts=True, key_added='rank_coarse')

In [ ]:
# Canonical-marker dotplot across coarse clusters -- the visual basis for the manual
# cluster->class assignment. Non-tubule classes are removed before re-integration.
coarse_panel = {**BROAD_TUBULE_MARKERS, **NON_TUBULE_MARKERS}
resolved_panel, missing_panel = available_markers_by_group(adata_cluster, coarse_panel, gene_lookup)
if missing_panel:
    print('Markers absent from gene space (skipped):', missing_panel)

sc.pl.dotplot(adata_cluster, resolved_panel, groupby='leiden_coarse', standard_scale='var', show=False)
plt.savefig(LEIDEN_EXPLORATION_DIR / 'coarse_marker_dotplot.png', dpi=150, bbox_inches='tight')
plt.show()

top_de = sc.get.rank_genes_groups_df(adata_cluster, group=None, key='rank_coarse')
top5 = (top_de.sort_values(['group', 'scores'], ascending=[True, False])
              .groupby('group').head(5).groupby('group')['names'].apply(lambda s: '; '.join(s)))
top5.to_csv(LEIDEN_EXPLORATION_DIR / 'coarse_cluster_top_markers.csv')
print(top5.to_string())

### Manual checkpoint — assign each coarse cluster to a class

Inspect `coarse_marker_dotplot.png` and the top markers above, then fill in `COARSE_LABELS`. Tubule
families (`PT, TL, TAL, DCT, CNT_CD`) are kept; `Glomerulus, Vessel, Stroma, SmoothMuscle, Immune` are
removed before the second Harmony pass.

In [ ]:
# MANUAL: map each coarse Leiden cluster id -> class. Edit after inspecting the dotplot.
COARSE_LABELS = {
    '0': 'PT', '1': 'TAL', '2': 'PT', '3': 'CNT_CD', '4': 'DCT', '5': 'Glomerulus', '6': 'Stroma', '7': 'CNT_CD', '8': 'None', '9': 'ATL/TAL', '10': 'DTL',
}
if not COARSE_LABELS:
    raise ValueError('Fill in COARSE_LABELS from the dotplot before proceeding.')

assign_cluster_labels(adata_cluster, 'leiden_coarse', COARSE_LABELS, 'coarse_class')
adata_all.obs['coarse_class'] = adata_cluster.obs['coarse_class'].reindex(adata_all.obs_names).to_numpy()
adata_all.obs['leiden_coarse'] = adata_cluster.obs['leiden_coarse'].reindex(adata_all.obs_names).astype(str).to_numpy()
sc.pl.embedding(adata_cluster, basis='umap', color=['leiden_coarse', 'coarse_class'], ncols=2, frameon=False)

In [ ]:
KEEP_TUBULE_CLASSES = ['PT', 'TL', 'TAL', 'DCT', 'CNT_CD', 'ATL/TAL', 'DTL']
REMOVE_CLASSES = ['Glomerulus', 'Vessel', 'Stroma', 'SmoothMuscle', 'Immune', 'None']

# Persist coarse_class back to the pass-1 file (ALL cells, incl. removed classes) so the
# spatial validation / QuPath steps can still use glomerulus + vessel locations.
adata_pass1 = sc.read_h5ad(HARMONY_OUTPUT_PATH)
adata_pass1.obs['coarse_class'] = adata_all.obs.loc[adata_pass1.obs_names, 'coarse_class'].to_numpy()
adata_pass1.write(HARMONY_OUTPUT_PATH)
print(f'Wrote coarse_class to {HARMONY_OUTPUT_PATH.name} (all {adata_pass1.n_obs:,} cells).')

keep_mask = adata_all.obs['coarse_class'].isin(KEEP_TUBULE_CLASSES).to_numpy()
for cls in REMOVE_CLASSES:
    print(f'  removing {int((adata_all.obs["coarse_class"] == cls).sum()):,} {cls} cells')
adata_tubule = adata_all[keep_mask].copy()
print(f'Tubule-only object: {adata_tubule.n_obs:,} tubules x {adata_tubule.n_vars:,} genes')

### Section 2c — Injury / failed-repair diagnostics

Score AKI injury markers on the tubule subset and profile them by cluster and by condition. Use this
to decide whether any low-identity ("Transitional"/unknown) cluster is injury-defined and should leave
the DPT continuum (set `EXCLUDE_INJURY_FROM_DPT` in Section 0). Decision deferred until after this
diagnostic.

In [ ]:
# # --- Section 2c: injury diagnostics ---
# tubule_lookup = build_gene_lookup(adata_tubule)
# injury_present = [tubule_lookup[g.upper()] for g in INJURY_MARKERS if g.upper() in tubule_lookup]
# print(f'Injury markers present ({len(injury_present)}/{len(INJURY_MARKERS)}): {injury_present}')
# sc.tl.score_genes(adata_tubule, injury_present, score_name='injury_score', use_raw=False)

# inj = adata_tubule.obs.groupby('leiden_coarse', observed=True).agg(
#     n=('injury_score', 'size'), injury_mean=('injury_score', 'mean')).reset_index()
# inj.to_csv(INJURY_DIAG_DIR / 'injury_by_cluster.csv', index=False)
# adata_tubule.obs[['sample', 'condition', 'leiden_coarse', 'coarse_class', 'injury_score']].to_csv(
#     INJURY_DIAG_DIR / 'injury_per_tubule.csv')

# fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
# sc.pl.embedding(adata_tubule, basis='umap', color='injury_score', cmap='magma',
#                 frameon=False, ax=axes[0], show=False)
# axes[0].set_title('Injury score (tubules)')
# cross = adata_tubule.obs.groupby(['coarse_class', 'condition'], observed=True)['injury_score'].mean().unstack()
# cross.plot(kind='bar', ax=axes[1]); axes[1].set_ylabel('mean injury_score')
# axes[1].set_title('Injury score by family x condition')
# fig.savefig(INJURY_DIAG_DIR / 'injury_diagnostics.png', dpi=150, bbox_inches='tight')
# plt.show()
# print(inj.sort_values('injury_mean', ascending=False).to_string(index=False))
# print('\nInspect injury_by_cluster.csv; set EXCLUDE_INJURY_FROM_DPT in Section 0 if warranted.')

### Section 2d — Re-integrate the tubule subset (pass 2), fine-type, and build DPT

In [ ]:
# Recompute INTERSECTION HVGs on the tubule subset, re-PCA, then sweep theta.
from pseudospace.harmony import select_harmony_hvgs_by_condition, run_harmony_rpy2, sweep_harmony_theta

adata_tubule = select_harmony_hvgs_by_condition(
    adata_tubule, group_key='condition', groups=('Control', 'IR'), mode='intersection',
    min_mean=HARMONY_HVG_MIN_MEAN, max_mean=HARMONY_HVG_MAX_MEAN, min_disp=HARMONY_HVG_MIN_DISP)
tub_hvg = 'highly_variable_for_harmony'
adata_tubule_hvg = adata_tubule[:, adata_tubule.var[tub_hvg]].copy()
adata_tubule_hvg.X = adata_tubule_hvg.layers['lognorm'].copy()
sc.tl.pca(adata_tubule_hvg, n_comps=HARMONY_PCA_N_COMPS)
adata_tubule.obsm['X_pca'] = adata_tubule_hvg.obsm['X_pca'].copy()
# print(f'Pass-2 HVGs (intersection, tubule-only): {int(adata_tubule.var[tub_hvg].sum())}')

# theta_sweep = sweep_harmony_theta(
#     adata_tubule_hvg, batch_key=BATCH_KEY, bio_label_key='coarse_class', thetas=HARMONY_THETA_GRID,
#     n_pcs=HARMONY_N_PCS, lambda_val=HARMONY_LAMBDA, max_iter=HARMONY_MAX_ITER, tau=HARMONY_TAU,
#     n_neighbors=HARMONY_NEIGHBORS_N, random_state=RANDOM_STATE)
# theta_sweep.to_csv(LEIDEN_EXPLORATION_DIR / 'harmony_theta_sweep.csv', index=False)
# display(theta_sweep)
# fig, ax = plt.subplots(figsize=(6, 4))
# ax.plot(theta_sweep['theta'], theta_sweep['batch_mixing_entropy'], 'o-', label='batch mixing (higher=mixed)')
# ax.plot(theta_sweep['theta'], theta_sweep['bio_conservation_silhouette'], 's-', label='bio conservation')
# ax.set_xlabel('theta'); ax.set_xscale('log', base=2); ax.legend()
# ax.set_title('Harmony theta sweep (pass 2)')
# fig.savefig(LEIDEN_EXPLORATION_DIR / 'harmony_theta_sweep.png', dpi=150, bbox_inches='tight'); plt.show()
# print(f'Pick the smallest theta that mixes samples without collapsing bio structure; '
#       f'set HARMONY_THETA in Section 0 (currently {HARMONY_THETA}).')

In [ ]:
# Pass-2 Harmony at the chosen theta -> new X_harmony (tubule-only embedding).
adata_tubule_hvg = run_harmony_rpy2(
    adata_tubule_hvg, batch_key=BATCH_KEY, n_pcs=HARMONY_N_PCS, theta=HARMONY_THETA,
    lambda_val=HARMONY_LAMBDA, max_iter=HARMONY_MAX_ITER, tau=HARMONY_TAU)
adata_tubule.obsm['X_harmony'] = adata_tubule_hvg.obsm['X_harmony'].copy()
sc.pp.neighbors(adata_tubule, use_rep='X_harmony', n_neighbors=HARMONY_NEIGHBORS_N, random_state=RANDOM_STATE)
sc.tl.umap(adata_tubule, min_dist=HARMONY_UMAP_MIN_DIST, spread=HARMONY_UMAP_SPREAD)
adata_tubule.write(PASS2_HARMONY_OUTPUT_PATH)
sc.pl.embedding(adata_tubule, basis='umap', color=['sample', 'condition', 'coarse_class'], ncols=3, frameon=False)
print(f'Saved pass-2 tubule-only Harmony object to {PASS2_HARMONY_OUTPUT_PATH.name}')

In [ ]:
# Fine nephron-segment clustering on the pass-2 embedding + per-resolution dotplots + seed stability.
from sklearn.metrics import adjusted_rand_score
fine_lookup = build_gene_lookup(adata_tubule)
seg_panel, seg_missing = available_markers_by_group(adata_tubule, SEGMENT_MARKERS, fine_lookup)
if seg_missing: print('Segment markers absent:', seg_missing)

for r in RESOLUTIONS_TO_EXPLORE:
    sc.tl.leiden(adata_tubule, resolution=r, key_added=f'leiden_res{r}',
                 flavor='igraph', n_iterations=2, directed=False, random_state=RANDOM_STATE)
    fig, ax = plt.subplots(figsize=(7, 6))
    sc.pl.embedding(adata_tubule, basis='umap', color=f'leiden_res{r}', frameon=False,
                    ax=ax, show=False, legend_loc='on data')
    fig.savefig(LEIDEN_EXPLORATION_DIR / f'fine_umap_res{r}.png', dpi=130, bbox_inches='tight'); plt.close(fig)
    sc.pl.dotplot(adata_tubule, seg_panel, groupby=f'leiden_res{r}', standard_scale='var', show=False)
    plt.savefig(LEIDEN_EXPLORATION_DIR / f'fine_dotplot_res{r}.png', dpi=130, bbox_inches='tight'); plt.close()
    print(LEIDEN_EXPLORATION_DIR / f'fine_umap_res{r}.png')

# Seed-ARI: robustness of each partition across Leiden seeds (review #5/#6).
seed_rows = []
for r in RESOLUTIONS_TO_EXPLORE:
    parts = {}
    for seed in [0, 1, 2]:
        sc.tl.leiden(adata_tubule, resolution=r, key_added='_seed', flavor='igraph',
                     n_iterations=2, directed=False, random_state=seed)
        parts[seed] = adata_tubule.obs['_seed'].astype(str).to_numpy(); del adata_tubule.obs['_seed']
    aris = [adjusted_rand_score(parts[a], parts[b]) for a, b in [(0, 1), (0, 2), (1, 2)]]
    seed_rows.append({'resolution': r, 'n_clusters': int(adata_tubule.obs[f'leiden_res{r}'].nunique()),
                      'seed_ari_mean': float(np.mean(aris))})
stability = pd.DataFrame(seed_rows)
stability.to_csv(LEIDEN_EXPLORATION_DIR / 'fine_resolution_stability.csv', index=False)
display(stability)

### Manual checkpoint — choose resolution and label fine clusters

Inspect `fine_dotplot_res*.png` / `fine_umap_res*.png` and `fine_resolution_stability.csv`, then set
`SELECTED_RESOLUTION` and fill in `FINE_LABELS` (cluster → broad family in `PT, TL, TAL, DCT, CNT_CD`;
unmapped → `Unknown`). PT S1/S2/S3 sub-structure is captured by pseudospace within PT, not by separate
labels.

In [ ]:
# # MANUAL: choose resolution and map each fine cluster -> broad family.
# SELECTED_RESOLUTION = 0.3   # <- set after inspecting fine_dotplot_res*.png
# if SELECTED_RESOLUTION is None:
#     raise ValueError('Set SELECTED_RESOLUTION from the fine dotplots before proceeding.')
# selected_cluster_col = f'leiden_res{SELECTED_RESOLUTION}'

# FINE_LABELS = {
#     '0': 'PT', '1': 'DCT', '2': 'PT', '3': 'CNT_CD', '4': 'TAL'
# }
# if not FINE_LABELS:
#     raise ValueError('Fill in FINE_LABELS from the fine dotplots before proceeding.')
# assign_cluster_labels(adata_tubule, selected_cluster_col, FINE_LABELS, 'broad_tubule_marker_call')

# sc.tl.rank_genes_groups(adata_tubule, selected_cluster_col, method='wilcoxon', pts=True, key_added='rank_fine')
# sc.get.rank_genes_groups_df(adata_tubule, group=None, key='rank_fine').to_csv(
#     LEIDEN_EXPLORATION_DIR / f'fine_cluster_de_res{SELECTED_RESOLUTION}.csv', index=False)

# # Per-cell broad-family argmax (fallback label + strip) and fine top-2 (diagnostic).
# broad_cols = []
# for fam, genes in BROAD_TUBULE_MARKERS.items():
#     present = [fine_lookup[g.upper()] for g in genes if g.upper() in fine_lookup]
#     if not present: continue
#     sc.tl.score_genes(adata_tubule, present, score_name=f'{fam}_broadscore', use_raw=False)
#     broad_cols.append((fam, f'{fam}_broadscore'))
# bmat = adata_tubule.obs[[c for _, c in broad_cols]].to_numpy()
# bnames = np.array([f for f, _ in broad_cols])
# adata_tubule.obs['broad_tubule_marker_call_percell'] = bnames[np.argmax(bmat, axis=1)]

# seg_cols = []
# for seg, genes in SEGMENT_MARKERS.items():
#     present = [fine_lookup[g.upper()] for g in genes if g.upper() in fine_lookup]
#     if not present: continue
#     sc.tl.score_genes(adata_tubule, present, score_name=f'{seg}_segscore', use_raw=False)
#     seg_cols.append((seg, f'{seg}_segscore'))
# smat = adata_tubule.obs[[c for _, c in seg_cols]].to_numpy()
# snames = np.array([s for s, _ in seg_cols])
# sorder = np.argsort(smat, axis=1)
# adata_tubule.obs['celltype_primary'] = snames[sorder[:, -1]]
# adata_tubule.obs['celltype_secondary'] = snames[sorder[:, -2]]
# adata_tubule.obs['celltype_margin'] = (np.take_along_axis(smat, sorder[:, -1:], 1).ravel()
#                                        - np.take_along_axis(smat, sorder[:, -2:-1], 1).ravel())

# sc.pl.embedding(adata_tubule, basis='umap', color='broad_tubule_marker_call', frameon=False)
# print(adata_tubule.obs['broad_tubule_marker_call'].value_counts())

In [ ]:
# --- Continuum: nephron tubules only. No fine re-clustering after the pass-2 Harmony --
# the PASS-1 `coarse_class` IS the family label. Downstream code (save_total_pseudotime_anndata,
# Section 3 heatmaps, subset DPT, Section 4) all keys off `broad_tubule_marker_call`, so alias
# coarse_class into that column here.
CONTINUUM_PREFIX = 'total'
adata_tubule.obs['broad_tubule_marker_call'] = adata_tubule.obs['coarse_class'].astype(str)

# Self-contained gene lookup: the fine-clustering cell above is now optional (exploration only).
tubule_lookup = build_gene_lookup(adata_tubule)

exclude_mask = adata_tubule.obs['broad_tubule_marker_call'].isin(['Unknown', 'nan', 'None', 'NaN']).to_numpy()
if EXCLUDE_INJURY_FROM_DPT and 'injury_score' in adata_tubule.obs:
    thr = np.nanpercentile(adata_tubule.obs['injury_score'], 90)
    exclude_mask = exclude_mask | (adata_tubule.obs['injury_score'].to_numpy() >= thr)
    print(f'Excluding Unknown + injury_score>=p90 ({thr:.3f}) from continuum')
adata_total = adata_tubule[~exclude_mask].copy()
print(f'TOTAL continuum: {adata_total.n_obs:,} tubules (excluded {int(exclude_mask.sum()):,})')
print(adata_total.obs['broad_tubule_marker_call'].value_counts())

for role in ['early', 'late']:
    genes = [tubule_lookup[g.upper()] for g in TOTAL_POSITION_MARKERS[role] if g.upper() in tubule_lookup]
    sc.tl.score_genes(adata_total, genes, score_name=f'{CONTINUUM_PREFIX}_{role}_trajectory_score', use_raw=False)
adata_total.obs[f'{CONTINUUM_PREFIX}_marker_axis'] = (
    adata_total.obs[f'{CONTINUUM_PREFIX}_late_trajectory_score'].astype(float)
    - adata_total.obs[f'{CONTINUUM_PREFIX}_early_trajectory_score'].astype(float))
adata_total.obs['total_segment_marker_call'] = adata_total.obs['coarse_class'].astype(str).values


In [ ]:
# Diffmap + PT-anchored root + Scanpy DPT on the PASS-2 embedding.
from pseudospace.trajectory import (orient_and_normalize, choose_root_global,
                                    choose_root_pt_cluster, choose_diffusion_component)
from scipy.stats import spearmanr
MIN_VALID_FOR_SPEARMAN = 100
EIGENVALUE_FLOOR = 0.5

marker_axis = adata_total.obs[f'{CONTINUUM_PREFIX}_marker_axis'].to_numpy(dtype=float)
use_rep = 'X_harmony'
n_neighbors = max(N_NEIGHBORS, int(np.sqrt(adata_total.n_obs)))
neighbors_key = 'trajectory_neighbors'
sc.pp.neighbors(adata_total, n_neighbors=n_neighbors, use_rep=use_rep,
                key_added=neighbors_key, random_state=RANDOM_STATE)
sc.tl.diffmap(adata_total, neighbors_key=neighbors_key)
best_component, component_df = choose_diffusion_component(
    adata_total, marker_axis, n_components_to_test=N_DIFFMAP_COMPONENTS_TO_TEST,
    eigenvalue_floor=EIGENVALUE_FLOOR, min_valid=MIN_VALID_FOR_SPEARMAN)
display(component_df)

# Root inside the PT family, at the lowest-marker-axis pass-1 Leiden cluster (falls back to
# coarse_class itself if leiden_coarse was not carried over).
root_cluster_col = 'leiden_coarse' if 'leiden_coarse' in adata_total.obs.columns else 'coarse_class'
iroot, root_cluster = choose_root_pt_cluster(adata_total, marker_axis, leiden_col=root_cluster_col,
                                             rep_key=use_rep, family_col='broad_tubule_marker_call')
if iroot is None:
    iroot = choose_root_global(marker_axis, adata_total, use_rep, bottom_quantile=0.01)
    root_cluster = 'fallback (global bottom-1%)'
adata_total.uns['iroot'] = iroot
print(f'Root: {adata_total.obs_names[iroot]} ({root_cluster_col}={root_cluster}), marker_axis={marker_axis[iroot]:.3f}')

sc.tl.dpt(adata_total, neighbors_key=neighbors_key)
total_scanpy_dpt = orient_and_normalize(adata_total.obs['dpt_pseudotime'].to_numpy(dtype=float),
                                        marker_axis, min_valid=MIN_VALID_FOR_SPEARMAN)
adata_total.obs['total_scanpy_dpt'] = total_scanpy_dpt
print('Spearman(total_scanpy_dpt, marker_axis) =',
      round(spearmanr(total_scanpy_dpt, marker_axis, nan_policy='omit').correlation, 3))
sc.pl.embedding(adata_total, basis='umap', color='total_scanpy_dpt', cmap='viridis', frameon=False,
                title='TOTAL Scanpy DPT (mouse-only, pass-2 embedding)')

In [ ]:
# Save the canonical DPT file (tubule-only; guarded against non-nephron / NaN leakage).
from pseudospace.trajectory import save_total_pseudotime_anndata
adata_dpt_saved = save_total_pseudotime_anndata(
    adata_total, 'total_scanpy_dpt', DPT_OUTPUT_PATH,
    expression_filtered_adata=adata_all_expression_filtered,
    annotation_adata=adata_total, project_dir=PROJECT_DIR,
    forbidden_labels=['Glomerulus', 'Glomeruli', 'Vessel', 'Stroma', 'SmoothMuscle', 'Immune', 'Unknown', 'nan', 'none'],
    assert_no_nan=True)

print('\n--- Section 2 summary ---')
print(f'Cell-type labels: pass-1 coarse_class (no fine re-clustering)   root cluster: {root_cluster}')
print(f'Tubule continuum: {adata_total.n_obs:,} tubules')
print(adata_total.obs['broad_tubule_marker_call'].value_counts())
print(f'total_scanpy_dpt range: [{np.nanmin(total_scanpy_dpt):.3f}, {np.nanmax(total_scanpy_dpt):.3f}]')

## Section 3 — Marker Heatmaps

Adapted from `3_heatmaps.ipynb`. Reuses the shared heatmap plotting function, marker panels, and
`recompute_subset_dpt` unchanged. Only `mouse_control` and `mouse_aki` subsets are produced -- there
are no human samples in this notebook, so no `human_*` outputs are generated.

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from scipy import sparse
from scipy.ndimage import gaussian_filter1d
from scipy.stats import spearmanr

plt.rcParams['figure.dpi'] = 120

OUTPUT_DIR = HEATMAP_OUTPUT_DIR  # plot_marker_heatmap saves here

adata_heatmap = sc.read_h5ad(DPT_OUTPUT_PATH)
if 'lognorm' in adata_heatmap.layers:
    adata_heatmap.X = adata_heatmap.layers['lognorm'].copy()

assert not adata_heatmap.obs['broad_tubule_marker_call'].isin(['Glomerulus', 'Vessel']).any(), (
    f'{DPT_OUTPUT_PATH.name} contains Glomerulus/Vessel cells -- this file should only contain '
    'tubule-family + Transitional cells. Re-run Section 2 (which now filters these out before '
    'saving the DPT AnnData) to regenerate it.'
)

# `broad_tubule_marker_call` here == the pass-1 coarse_class labels (see Section 2d).
# `broad_tubule_marker_call_percell` is only produced by the (now skipped) fine-clustering
# cell, so it is not required.
BASE_OBS_COLUMNS = [
    'sample', 'species', 'condition',
    'broad_tubule_marker_call',
    'total_segment_marker_call', 'total_scanpy_dpt',
]
missing_obs = [col for col in BASE_OBS_COLUMNS if col not in adata_heatmap.obs.columns]
if missing_obs:
    raise KeyError(f'{DPT_OUTPUT_PATH.name} is missing required obs columns: {missing_obs}')

print(f'Loaded {DPT_OUTPUT_PATH.name}: {adata_heatmap.n_obs:,} tubules x {adata_heatmap.n_vars:,} genes')
print(f'Heatmaps will be saved to: {HEATMAP_OUTPUT_DIR}')

In [ ]:
# ==============================================================================
# PROXIMAL TUBULE POSITIONAL MARKERS — DERIVED FROM THE PROVIDED CHART
# ==============================================================================
# Assignment:
#   S1-only or S1 -> S2  => PT-S1
#   S2-only or S2 -> S3  => PT-S2
#   S3-only              => PT-S3
#   S1 -> S2 -> S3       => PT_PAN_MARKERS
PT_MARKER_GROUPS = {
    'PT-S1': [
        {
            'label': 'SLC5A2 / Slc5a2',
            'candidates': ['SLC5A2', 'Slc5a2', 'slc5a2'],
        },
    ],
    'PT-S2': [
        {
            'label': 'SLC1A1 / Slc1a1',
            'candidates': ['SLC1A1', 'Slc1a1', 'slc1a1'],
        },
        {
            'label': 'SLC7A13 / Slc7a13',
            'candidates': ['SLC7A13', 'Slc7a13', 'slc7a13'],
        },
        {
            'label': 'SLC13A1 / Slc13a1',
            'candidates': ['SLC13A1', 'Slc13a1', 'slc13a1'],
        },
        {
            'label': 'SLC13A3 / Slc13a3',
            'candidates': ['SLC13A3', 'Slc13a3', 'slc13a3'],
        },
        {
            'label': 'SLC22A2 / Slc22a2',
            'candidates': ['SLC22A2', 'Slc22a2', 'slc22a2'],
        },
    ],
    'PT-S3': [
        {
            'label': 'SLC5A1 / Slc5a1',
            'candidates': ['SLC5A1', 'Slc5a1', 'slc5a1'],
        },
        {
            'label': 'SLC6A13 / Slc6a13',
            'candidates': ['SLC6A13', 'Slc6a13', 'slc6a13'],
        },
    ],
}

PT_GROUP_ORDER = ['PT-S1', 'PT-S2', 'PT-S3']

PT_GROUP_COLORS = {
    'PT-S1': '#4C9BD3',
    'PT-S2': '#5B8E55',
    'PT-S3': '#D6B48A',
}


# PT identity markers that do not resolve S1/S2/S3.
PT_PAN_MARKERS = [
    {
        'label': 'SLC3A1 / Slc3a1',
        'candidates': ['SLC3A1', 'Slc3a1', 'slc3a1'],
    },
    {
        'label': 'SLC4A4 / Slc4a4',
        'candidates': ['SLC4A4', 'Slc4a4', 'slc4a4'],
    },
    {
        'label': 'SLC5A9 / Slc5a9',
        'candidates': ['SLC5A9', 'Slc5a9', 'slc5a9'],
    },
    {
        'label': 'SLC6A14 / Slc6a14',
        'candidates': ['SLC6A14', 'Slc6a14', 'slc6a14'],
    },
    {
        'label': 'SLC6A19 / Slc6a19',
        'candidates': ['SLC6A19', 'Slc6a19', 'slc6a19'],
    },
    {
        'label': 'SLC7A8 / Slc7a8',
        'candidates': ['SLC7A8', 'Slc7a8', 'slc7a8'],
    },
    {
        'label': 'SLC9A3 / Slc9a3',
        'candidates': ['SLC9A3', 'Slc9a3', 'slc9a3'],
    },
    {
        'label': 'SLC22A5 / Slc22a5',
        'candidates': ['SLC22A5', 'Slc22a5', 'slc22a5'],
    },
]


# ==============================================================================
# THIN-LIMB SUBSEGMENTS — CHEN ET AL., JASN 2021
# ==============================================================================
THIN_LIMB_MARKER_GROUPS = {
    'DTL1': [
        {
            'label': 'CORIN / Corin',
            'candidates': ['CORIN', 'Corin', 'corin'],
        },
        {
            'label': 'UNCX / Uncx',
            'candidates': ['UNCX', 'Uncx', 'uncx'],
        },
    ],
    'DTL2': [
        {
            'label': 'FST / Fst',
            'candidates': ['FST', 'Fst', 'fst'],
        },
        {
            'label': 'CDH13 / Cdh13',
            'candidates': ['CDH13', 'Cdh13', 'cdh13'],
        },
        {
            'label': 'STK32A / Stk32a',
            'candidates': ['STK32A', 'Stk32a', 'stk32a'],
        },
    ],
    'DTL3': [
        {
            'label': 'NR2E3 / Nr2e3',
            'candidates': ['NR2E3', 'Nr2e3', 'nr2e3'],
        },
        {
            'label': 'DMKN / Dmkn',
            'candidates': ['DMKN', 'Dmkn', 'dmkn'],
        },
    ],
    'ATL': [
        {
            'label': 'IGFBP2 / Igfbp2',
            'candidates': ['IGFBP2', 'Igfbp2', 'igfbp2'],
        },
        {
            'label': 'PSCA / Psca',
            'candidates': ['PSCA', 'Psca', 'psca'],
        },
        {
            'label': 'SPTSSB / Sptssb',
            'candidates': ['SPTSSB', 'Sptssb', 'sptssb'],
        },
        {
            'label': 'CCDC3 / Ccdc3',
            'candidates': ['CCDC3', 'Ccdc3', 'ccdc3'],
        },
        {
            'label': 'CLCNKA / Clcnka',
            'candidates': ['CLCNKA', 'Clcnka', 'clcnka'],
        },
    ],
}

THIN_LIMB_GROUP_ORDER = ['DTL1', 'DTL2', 'DTL3', 'ATL']

THIN_LIMB_GROUP_COLORS = {
    'DTL1': '#8AC926',
    'DTL2': '#55A630',
    'DTL3': '#2B9348',
    'ATL': '#007F5F',
}


# ==============================================================================
# MEDULLARY VERSUS CORTICAL TAL
# ==============================================================================
# Sycn and Shd were experimentally validated as regional markers.
# Tmem86a and Dapk1 were listed as region-enriched in Table 1.
TAL_MARKER_GROUPS = {
    'mTAL': [
        {
            'label': 'SYCN / Sycn',
            'candidates': ['SYCN', 'Sycn', 'sycn'],
        },
        {
            'label': 'TMEM86A / Tmem86a',
            'candidates': ['TMEM86A', 'Tmem86a', 'tmem86a'],
        },
    ],
    'cTAL': [
        {
            'label': 'SHD / Shd',
            'candidates': ['SHD', 'Shd', 'shd'],
        },
        {
            'label': 'DAPK1 / Dapk1',
            'candidates': ['DAPK1', 'Dapk1', 'dapk1'],
        },
    ],
}

TAL_GROUP_ORDER = ['mTAL', 'cTAL']

TAL_GROUP_COLORS = {
    'mTAL': '#4C78A8',
    'cTAL': '#F58518',
}


# TAL identity markers that do not separate mTAL from cTAL.
TAL_PAN_MARKERS = [
    {
        'label': 'SLC12A1 / Slc12a1',
        'candidates': ['SLC12A1', 'Slc12a1', 'slc12a1'],
    },
    {
        'label': 'UMOD / Umod',
        'candidates': ['UMOD', 'Umod', 'umod'],
    },
]


# ==============================================================================
# DCT1 VERSUS DCT2
# ==============================================================================
# Both groups must first satisfy the pan-DCT Slc12a3 signal.
#
# DCT1: Pvalb positive.
# DCT2: Pvalb absent/low and Trpv5 elevated.
#
# Trpv5 is DCT2-enriched, not absolutely DCT2-specific.
DCT_MARKER_GROUPS = {
    'DCT1': [
        {
            'label': 'PVALB / Pvalb',
            'candidates': ['PVALB', 'Pvalb', 'pvalb'],
        },
    ],
    'DCT2': [
        {
            'label': 'TRPV5 / Trpv5',
            'candidates': ['TRPV5', 'Trpv5', 'trpv5'],
        },
    ],
}

DCT_GROUP_ORDER = ['DCT1', 'DCT2']

DCT_GROUP_COLORS = {
    'DCT1': '#4C78A8',
    'DCT2': '#F58518',
}


DCT_PAN_MARKERS = [
    {
        'label': 'SLC12A3 / Slc12a3',
        'candidates': ['SLC12A3', 'Slc12a3', 'slc12a3'],
    },
]


# Do not score these as uniquely positive for DCT2 because they also occur
# in DCT1 and/or CNT. They are supporting evidence only.
DCT2_SUPPORT_MARKERS = [
    {
        'label': 'CALB1 / Calb1',
        'candidates': ['CALB1', 'Calb1', 'calb1'],
    },
    {
        'label': 'S100G / S100g',
        'candidates': ['S100G', 'S100g', 's100g'],
    },
    {
        'label': 'SLC8A1 / Slc8a1',
        'candidates': ['SLC8A1', 'Slc8a1', 'slc8a1'],
    },
]


DCT_CLASSIFICATION_RULES = {
    'DCT1': {
        'required_positive': ['SLC12A3', 'PVALB'],
        'expected_negative': [],
    },
    'DCT2': {
        'required_positive': ['SLC12A3'],
        'support_positive': ['TRPV5', 'CALB1', 'S100G', 'SLC8A1'],
        'expected_negative': ['PVALB'],
    },
}


# ==============================================================================
# CNT AND COLLECTING-DUCT REGIONS
# ==============================================================================
# Apply this panel only after establishing CNT/CD identity.
#
# Aqp2 and Aqp4 span multiple collecting regions and are therefore kept in
# COLLECTING_REGION_PAN_MARKERS rather than assigned to one region.
COLLECTING_REGION_MARKER_GROUPS = {
    'CNT': [
        {
            'label': 'CALB1 / Calb1',
            'candidates': ['CALB1', 'Calb1', 'calb1'],
        },
        {
            'label': 'S100G / S100g',
            'candidates': ['S100G', 'S100g', 's100g'],
        },
    ],
    'CCD': [
        {
            'label': 'ATP4A / Atp4a',
            'candidates': ['ATP4A', 'Atp4a', 'atp4a'],
        },
    ],
    'OMCD': [
        {
            'label': 'HSD11B2 / Hsd11b2',
            'candidates': ['HSD11B2', 'Hsd11b2', 'hsd11b2'],
        },
    ],
    'IMCD': [
        {
            'label': 'WNT7B / Wnt7b',
            'candidates': ['WNT7B', 'Wnt7b', 'wnt7b'],
        },
        {
            'label': 'WNT4 / Wnt4',
            'candidates': ['WNT4', 'Wnt4', 'wnt4'],
        },
        {
            'label': 'PAX2 / Pax2',
            'candidates': ['PAX2', 'Pax2', 'pax2'],
        },
    ],
}

COLLECTING_REGION_GROUP_ORDER = ['CNT', 'CCD', 'OMCD', 'IMCD']

COLLECTING_REGION_GROUP_COLORS = {
    'CNT': '#76B7B2',
    'CCD': '#4E79A7',
    'OMCD': '#9C755F',
    'IMCD': '#593C8F',
}


COLLECTING_REGION_PAN_MARKERS = [
    {
        'label': 'AQP2 / Aqp2',
        'candidates': ['AQP2', 'Aqp2', 'aqp2'],
    },
    {
        'label': 'AQP4 / Aqp4',
        'candidates': ['AQP4', 'Aqp4', 'aqp4'],
    },
]


# ==============================================================================
# COLLECTING-DUCT CELL IDENTITIES
# ==============================================================================
# This is an orthogonal cell-type split, not a cortex-to-medulla position split.
COLLECTING_CELLTYPE_MARKER_GROUPS = {
    'CD-PC': [
        {
            'label': 'AQP2 / Aqp2',
            'candidates': ['AQP2', 'Aqp2', 'aqp2'],
        },
        {
            'label': 'AQP3 / Aqp3',
            'candidates': ['AQP3', 'Aqp3', 'aqp3'],
        },
        {
            'label': 'SCNN1G / Scnn1g',
            'candidates': ['SCNN1G', 'Scnn1g', 'scnn1g'],
        },
    ],
    'CD-A-IC': [
        {
            'label': 'SLC4A1 / Slc4a1',
            'candidates': ['SLC4A1', 'Slc4a1', 'slc4a1'],
        },
        {
            'label': 'AQP6 / Aqp6',
            'candidates': ['AQP6', 'Aqp6', 'aqp6'],
        },
    ],
    'CD-B-IC': [
        {
            'label': 'SLC26A4 / Slc26a4',
            'candidates': ['SLC26A4', 'Slc26a4', 'slc26a4'],
        },
        {
            'label': 'INSRR / Insrr',
            'candidates': ['INSRR', 'Insrr', 'insrr'],
        },
    ],
}

COLLECTING_CELLTYPE_GROUP_ORDER = ['CD-PC', 'CD-A-IC', 'CD-B-IC']

COLLECTING_CELLTYPE_GROUP_COLORS = {
    'CD-PC': '#4E79A7',
    'CD-A-IC': '#E15759',
    'CD-B-IC': '#B07AA1',
}


# These identify IC identity but do not distinguish A-IC from B-IC.
IC_PAN_MARKERS = [
    {
        'label': 'FOXI1 / Foxi1',
        'candidates': ['FOXI1', 'Foxi1', 'foxi1'],
    },
    {
        'label': 'ATP6V1B1 / Atp6v1b1',
        'candidates': ['ATP6V1B1', 'Atp6v1b1', 'atp6v1b1'],
    },
    {
        'label': 'ATP6V1G3 / Atp6v1g3',
        'candidates': ['ATP6V1G3', 'Atp6v1g3', 'atp6v1g3'],
    },
]


# ==============================================================================
# CONTINUOUS AXIS MARKERS
# ==============================================================================
# One non-repeated representative marker per positional segment.
#
# DCT2 is a contrast-based call:
#   Slc12a3 positive, Pvalb low/absent, Trpv5 elevated.
#
# CNT should likewise be confirmed by an Aqp2 + Calb1/S100g combination.
TOTAL_NEPHRON_MARKERS = [
    {'label': 'Nphs1 (Glom)', 'candidates': ['NPHS1', 'Nphs1', 'nephrin']},
    {'label': 'Nphs2 (Glom)', 'candidates': ['NPHS2', 'Nphs2', 'podocin']},

    {'label': 'Slc5a2 (PT-S1)', 'candidates': ['SLC5A2', 'Slc5a2', 'slc5a2']},
    {'label': 'Slc1a1 (PT-S2)', 'candidates': ['SLC1A1', 'Slc1a1', 'slc1a1']},
    {'label': 'Slc5a1 (PT-S3)', 'candidates': ['SLC5A1', 'Slc5a1', 'slc5a1']},

    {'label': 'Corin (DTL1)', 'candidates': ['CORIN', 'Corin', 'corin']},
    {'label': 'Cdh13 (DTL2)', 'candidates': ['CDH13', 'Cdh13', 'cdh13']},
    {'label': 'Nr2e3 (DTL3)', 'candidates': ['NR2E3', 'Nr2e3', 'nr2e3']},
    {'label': 'Sptssb (ATL)', 'candidates': ['SPTSSB', 'Sptssb', 'sptssb']},

    {'label': 'Sycn (mTAL)', 'candidates': ['SYCN', 'Sycn', 'sycn']},
    {'label': 'Shd (cTAL)', 'candidates': ['SHD', 'Shd', 'shd']},

    {'label': 'Pvalb (DCT1)', 'candidates': ['PVALB', 'Pvalb', 'pvalb']},
    {'label': 'Trpv5 (DCT2-enriched)', 'candidates': ['TRPV5', 'Trpv5', 'trpv5']},

    {'label': 'Calb1 (CNT; confirm with Aqp2)', 'candidates': ['CALB1', 'Calb1', 'calb1']},
    {'label': 'Atp4a (CCD-enriched)', 'candidates': ['ATP4A', 'Atp4a', 'atp4a']},
    {'label': 'Hsd11b2 (OMCD-enriched)', 'candidates': ['HSD11B2', 'Hsd11b2', 'hsd11b2']},
    {'label': 'Wnt7b (IMCD-enriched)', 'candidates': ['WNT7B', 'Wnt7b', 'wnt7b']},
]


# ==============================================================================
# PROVENANCE
# ==============================================================================
MARKER_PROVENANCE = {
    'PT': {
        'source': 'Provided nephron-segment expression chart',
        'location': 'PS1, PS2, and PS3 columns',
    },
    'DTL1_DTL2_DTL3_ATL': {
        'paper': (
            'A Comprehensive Map of mRNAs and Their Isoforms across '
            'All 14 Renal Tubule Segments of Mouse'
        ),
        'location': (
            'Results paragraph describing segment-specific genes among '
            'DTL1, DTL2, DTL3, and ATL'
        ),
        'url': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC8017530/',
    },
    'mTAL_cTAL': {
        'paper': 'Atlas of Gene Expression in the Mouse Kidney',
        'location': (
            'Table 1 and validation of Sycn as mTAL-enriched and '
            'Shd as cTAL-enriched'
        ),
        'url': (
            'https://journals.physiology.org/doi/pdf/'
            '10.1152/physiolgenomics.00093.2010'
        ),
    },
    'DCT1_DCT2': {
        'paper': (
            'Targeted Single-Cell RNA-seq Identifies Minority Cell Types '
            'of Kidney Distal Nephron'
        ),
        'location': 'Results and Figures 4-5',
        'url': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC8017539/',
    },
    'CNT_CCD_OMCD_IMCD': {
        'paper': (
            'A Comprehensive Map of mRNAs and Their Isoforms across '
            'All 14 Renal Tubule Segments of Mouse'
        ),
        'location': (
            'Supplemental Figure 1B cluster-marker table and the main-text '
            'description of IMCD-enriched genes'
        ),
        'url': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC8017530/',
        'supplement_url': (
            'https://cdn-links.lww.com/permalink/jsn/d/'
            'jsn_32_4_2022_12_09_chen_asn2020101406_sdc1.pdf'
        ),
    },
    'CD_PC_AIC_BIC': {
        'paper': (
            'Transcriptomes of Major Renal Collecting Duct Cell Types '
            'in Mouse Identified by Single-Cell RNA-seq'
        ),
        'location': 'Figure 5C and Dataset S4',
        'url': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC5699061/',
    },
}


# ==============================================================================
# VALIDATION
# ==============================================================================
def _canonical_gene(marker):
    return marker['candidates'][0].upper()


def _validate_no_repeated_assignments(grouped_markers, panel_name):
    seen = {}

    for group, markers in grouped_markers.items():
        for marker in markers:
            gene = _canonical_gene(marker)

            if gene in seen:
                raise ValueError(
                    f'{gene} is assigned to both {seen[gene]!r} and '
                    f'{group!r} in {panel_name}.'
                )

            seen[gene] = group

    return seen


_validate_no_repeated_assignments(
    PT_MARKER_GROUPS,
    'PT_MARKER_GROUPS',
)
_validate_no_repeated_assignments(
    THIN_LIMB_MARKER_GROUPS,
    'THIN_LIMB_MARKER_GROUPS',
)
_validate_no_repeated_assignments(
    TAL_MARKER_GROUPS,
    'TAL_MARKER_GROUPS',
)
_validate_no_repeated_assignments(
    DCT_MARKER_GROUPS,
    'DCT_MARKER_GROUPS',
)
_validate_no_repeated_assignments(
    COLLECTING_REGION_MARKER_GROUPS,
    'COLLECTING_REGION_MARKER_GROUPS',
)
_validate_no_repeated_assignments(
    COLLECTING_CELLTYPE_MARKER_GROUPS,
    'COLLECTING_CELLTYPE_MARKER_GROUPS',
)

print(
    'Marker panels defined with non-repeated assignments; '
    'pan-segment markers and contrast-based calls are stored separately.'
)

In [ ]:
from pseudospace.markers import resolve_available_marker_groups
from pseudospace.heatmaps import plot_marker_heatmap

In [ ]:
mouse_control_mask = adata_heatmap.obs['condition'].astype(str).str.lower().eq('control').to_numpy()
mouse_aki_mask = adata_heatmap.obs['condition'].astype(str).str.lower().eq('ir').to_numpy()

print(f'mouse_control: {int(mouse_control_mask.sum()):,} tubules')
print(f'mouse_aki: {int(mouse_aki_mask.sum()):,} tubules')

segment_specs = {
    'PT': {'markers': PT_MARKER_GROUPS, 'order': PT_GROUP_ORDER, 'colors': PT_GROUP_COLORS, 'z_clip': 2},
    'TAL': {'markers': TAL_MARKER_GROUPS, 'order': TAL_GROUP_ORDER, 'colors': TAL_GROUP_COLORS, 'z_clip': 2},
    'DCT': {'markers': DCT_MARKER_GROUPS, 'order': DCT_GROUP_ORDER, 'colors': DCT_GROUP_COLORS, 'z_clip': 2},
}

TOTAL_SEGMENT_COLORS = {
    'PT': '#4C9BD3', 'TL': '#7E57C2', 'CNT_CD': '#8D6E63',
    'PT-S1': '#4C9BD3', 'PT-S2': '#5B8E55', 'PT-S3': '#D6B48A',
    'DTL': '#7E57C2', 'ATL': '#26A69A', 'TAL': '#F58518',
    'DCT': '#D81B60', 'CNT': '#8D6E63', 'CD': '#546E7A',
}
TOTAL_MARKER_GROUPS = {
    row['label']: [{'label': row['label'], 'candidates': row['candidates']}]
    for row in TOTAL_NEPHRON_MARKERS
}
TOTAL_GROUP_ORDER = [row['label'] for row in TOTAL_NEPHRON_MARKERS]
TOTAL_GROUP_COLORS = {group: TOTAL_SEGMENT_COLORS.get(group.split()[0], '#4D4D4D') for group in TOTAL_GROUP_ORDER}

### Marker resolution check

Confirms every panel resolves against the native mouse gene space before any plotting runs.

In [ ]:
panels_to_check = {
    'TOTAL': (TOTAL_MARKER_GROUPS, TOTAL_GROUP_ORDER),
    'PT': (PT_MARKER_GROUPS, PT_GROUP_ORDER),
    'TAL': (TAL_MARKER_GROUPS, TAL_GROUP_ORDER),
    'DCT': (DCT_MARKER_GROUPS, DCT_GROUP_ORDER),
}

any_missing = False
for panel_name, (marker_groups, group_order) in panels_to_check.items():
    _, _, _, missing = resolve_available_marker_groups(adata_heatmap, marker_groups, group_order)
    n_total = sum(len(v) for v in marker_groups.values())
    status = f'{panel_name}: {n_total - len(missing)}/{n_total} markers resolved'
    if missing:
        status += f'; MISSING: {missing}'
        any_missing = True
    print(status)

if any_missing:
    warnings.warn('One or more marker panels are missing genes in the mouse-only gene space.')
else:
    print('\nAll marker panels resolved with no missing genes.')

### Heatmaps on the Global Nephron Coordinate

In [ ]:
total_heatmap_results = {}

total_heatmap_results['Mouse_Control_TOTAL'] = plot_marker_heatmap(
    adata_heatmap, TOTAL_MARKER_GROUPS, TOTAL_GROUP_ORDER, TOTAL_GROUP_COLORS,
    output_dir=OUTPUT_DIR, project_dir=PROJECT_DIR,
    pseudotime_col='total_scanpy_dpt', cluster_col='total_segment_marker_call', cluster_value=None,
    mask=mouse_control_mask, target_label='Mouse Control', strip_col='total_segment_marker_call',
    strip_colors=TOTAL_SEGMENT_COLORS,
    title='Mouse Control nephron markers across total cluster-defined Scanpy DPT',
    output_name='mouse_control_total_marker_heatmap.png',
    n_bins=N_BINS, smooth_sigma=SMOOTH_SIGMA, trim_fraction=TRIM_FRACTION, z_clip=3,
    show_module_legend=False, show_group_labels=False, figsize=(18, 11),
)

total_heatmap_results['Mouse_AKI_TOTAL'] = plot_marker_heatmap(
    adata_heatmap, TOTAL_MARKER_GROUPS, TOTAL_GROUP_ORDER, TOTAL_GROUP_COLORS,
    output_dir=OUTPUT_DIR, project_dir=PROJECT_DIR,
    pseudotime_col='total_scanpy_dpt', cluster_col='total_segment_marker_call', cluster_value=None,
    mask=mouse_aki_mask, target_label='Mouse AKI', strip_col='total_segment_marker_call',
    strip_colors=TOTAL_SEGMENT_COLORS,
    title='Mouse AKI nephron markers across total cluster-defined Scanpy DPT',
    output_name='mouse_aki_total_marker_heatmap.png',
    n_bins=N_BINS, smooth_sigma=SMOOTH_SIGMA, trim_fraction=TRIM_FRACTION, z_clip=3,
    show_module_legend=False, show_group_labels=False, figsize=(18, 11),
)

total_summary = pd.DataFrame(total_heatmap_results).T
total_summary.index.name = 'heatmap'
total_summary.to_csv(OUTPUT_DIR / 'total_nephron_heatmap_summary.csv')
display(total_summary[['n_cells_before_trim', 'n_cells_trimmed_per_end', 'n_cells', 'cluster_col', 'cluster_value', 'pseudotime_col', 'dpt_min', 'dpt_max', 'output_path']])

In [ ]:
# Unsmoothed companion (smooth_sigma=0) -- guards against smoothing-manufactured monotonicity.
plot_marker_heatmap(
    adata_heatmap, TOTAL_MARKER_GROUPS, TOTAL_GROUP_ORDER, TOTAL_GROUP_COLORS,
    output_dir=OUTPUT_DIR, project_dir=PROJECT_DIR,
    pseudotime_col='total_scanpy_dpt', cluster_col='total_segment_marker_call', cluster_value=None,
    mask=mouse_control_mask, target_label='Mouse Control (unsmoothed)',
    strip_col='total_segment_marker_call', strip_colors=TOTAL_SEGMENT_COLORS,
    title='Mouse Control nephron markers across total DPT (UNSMOOTHED)',
    output_name='mouse_control_total_marker_heatmap_unsmoothed.png',
    n_bins=N_BINS, smooth_sigma=0, trim_fraction=TRIM_FRACTION, z_clip=3,
    show_module_legend=False, show_group_labels=False, figsize=(18, 11))

### PT, TAL, and DCT on Global Pseudospace

In [ ]:
global_segment_results = {}

for condition_key, condition_label, condition_mask in [
    ('Mouse_Control', 'Mouse Control', mouse_control_mask),
    ('Mouse_AKI', 'Mouse AKI', mouse_aki_mask),
]:
    for segment, spec in segment_specs.items():
        global_segment_results[f'{condition_key}_{segment}'] = plot_marker_heatmap(
            adata_heatmap, spec['markers'], spec['order'], spec['colors'],
        output_dir=OUTPUT_DIR, project_dir=PROJECT_DIR,
            pseudotime_col='total_scanpy_dpt', cluster_col='broad_tubule_marker_call', cluster_value=segment,
            mask=condition_mask, target_label=condition_label,
            title=f'{condition_label} {segment} markers on global nephron Scanpy DPT',
            output_name=f'{condition_key.lower()}_{segment.lower()}_global_dpt_marker_heatmap.png',
            n_bins=N_BINS, smooth_sigma=SMOOTH_SIGMA, trim_fraction=TRIM_FRACTION, z_clip=spec['z_clip'],
        )

global_segment_summary = pd.DataFrame(global_segment_results).T
global_segment_summary.index.name = 'heatmap'
global_segment_summary.to_csv(OUTPUT_DIR / 'global_segment_heatmap_summary.csv')
display(global_segment_summary[['n_cells_before_trim', 'n_cells_trimmed_per_end', 'n_cells', 'cluster_value', 'pseudotime_col', 'dpt_min', 'dpt_max', 'output_path']])

### Cluster-Specific Diffusion Pseudotime

Reconstructs diffusion pseudotime from scratch within each broad Leiden-derived PT, TAL, and DCT
subset. The resulting in-memory columns are `pt_subset_scanpy_dpt`, `tal_subset_scanpy_dpt`, and
`dct_subset_scanpy_dpt`. They do not overwrite the global DPT coordinate or the saved AnnData file.

In [ ]:
from pseudospace.trajectory import recompute_subset_dpt


SUBSET_DPT_SPECS = {
    'PT': {'marker_groups': PT_MARKER_GROUPS, 'group_order': PT_GROUP_ORDER, 'pseudotime_col': 'pt_subset_scanpy_dpt'},
    'TAL': {'marker_groups': TAL_MARKER_GROUPS, 'group_order': TAL_GROUP_ORDER, 'pseudotime_col': 'tal_subset_scanpy_dpt'},
    'DCT': {'marker_groups': DCT_MARKER_GROUPS, 'group_order': DCT_GROUP_ORDER, 'pseudotime_col': 'dct_subset_scanpy_dpt'},
}


subset_dpt_diagnostics = []
for segment, spec in SUBSET_DPT_SPECS.items():
    subset_dpt_diagnostics.append(
        recompute_subset_dpt(
            adata_heatmap, segment=segment, marker_groups=spec['marker_groups'],
            group_order=spec['group_order'], output_col=spec['pseudotime_col'],
        )
    )

subset_dpt_diagnostics = pd.DataFrame(subset_dpt_diagnostics)
subset_dpt_diagnostics.to_csv(OUTPUT_DIR / 'subset_recomputed_dpt_diagnostics.csv', index=False)
display(subset_dpt_diagnostics)

In [ ]:
fig, axes = plt.subplots(len(SUBSET_DPT_SPECS), 2, figsize=(12, 4 * len(SUBSET_DPT_SPECS)), constrained_layout=True)

for row_i, (segment, spec) in enumerate(SUBSET_DPT_SPECS.items()):
    subset_mask = adata_heatmap.obs['broad_tubule_marker_call'].astype(str).eq(segment).to_numpy()
    umap = np.asarray(adata_heatmap.obsm['X_umap'])[subset_mask]
    subset_dpt = adata_heatmap.obs.loc[subset_mask, spec['pseudotime_col']].to_numpy(dtype=float)
    total_dpt = adata_heatmap.obs.loc[subset_mask, 'total_scanpy_dpt'].to_numpy(dtype=float)

    axes[row_i, 0].scatter(
        np.asarray(adata_heatmap.obsm['X_umap'])[:, 0], np.asarray(adata_heatmap.obsm['X_umap'])[:, 1],
        s=1, color='#E0E0E0', alpha=0.2, linewidths=0, rasterized=True,
    )
    scatter = axes[row_i, 0].scatter(umap[:, 0], umap[:, 1], c=subset_dpt, cmap='viridis', vmin=0, vmax=1, s=4, linewidths=0, rasterized=True)
    axes[row_i, 0].set_title(f'{segment}: subset-recomputed DPT')
    axes[row_i, 0].set_xticks([])
    axes[row_i, 0].set_yticks([])
    fig.colorbar(scatter, ax=axes[row_i, 0], label=spec['pseudotime_col'])

    finite = np.isfinite(total_dpt) & np.isfinite(subset_dpt)
    axes[row_i, 1].hexbin(total_dpt[finite], subset_dpt[finite], gridsize=55, mincnt=1, cmap='magma')
    corr = spearmanr(total_dpt[finite], subset_dpt[finite]).correlation
    axes[row_i, 1].set_title(f'{segment}: total vs subset DPT (Spearman={corr:.2f})')
    axes[row_i, 1].set_xlabel('total_scanpy_dpt')
    axes[row_i, 1].set_ylabel(spec['pseudotime_col'])

fig.savefig(OUTPUT_DIR / 'subset_recomputed_dpt_diagnostics_umap.png', dpi=220, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(len(SUBSET_DPT_SPECS), 2, figsize=(12, 4 * len(SUBSET_DPT_SPECS)), constrained_layout=True)

for row_i, (segment, spec) in enumerate(SUBSET_DPT_SPECS.items()):
    subset_mask = adata_heatmap.obs['broad_tubule_marker_call'].astype(str).eq(segment).to_numpy()
    umap = np.asarray(adata_heatmap.obsm['X_pca'])[subset_mask]
    subset_dpt = adata_heatmap.obs.loc[subset_mask, spec['pseudotime_col']].to_numpy(dtype=float)
    total_dpt = adata_heatmap.obs.loc[subset_mask, 'total_scanpy_dpt'].to_numpy(dtype=float)

    axes[row_i, 0].scatter(
        np.asarray(adata_heatmap.obsm['X_pca'])[:, 0], np.asarray(adata_heatmap.obsm['X_pca'])[:, 1],
        s=1, color='#E0E0E0', alpha=0.2, linewidths=0, rasterized=True,
    )
    scatter = axes[row_i, 0].scatter(umap[:, 0], umap[:, 1], c=subset_dpt, cmap='viridis', vmin=0, vmax=1, s=4, linewidths=0, rasterized=True)
    axes[row_i, 0].set_title(f'{segment}: subset-recomputed DPT')
    axes[row_i, 0].set_xticks([])
    axes[row_i, 0].set_yticks([])
    fig.colorbar(scatter, ax=axes[row_i, 0], label=spec['pseudotime_col'])

    finite = np.isfinite(total_dpt) & np.isfinite(subset_dpt)
    axes[row_i, 1].hexbin(total_dpt[finite], subset_dpt[finite], gridsize=55, mincnt=1, cmap='magma')
    corr = spearmanr(total_dpt[finite], subset_dpt[finite]).correlation
    axes[row_i, 1].set_title(f'{segment}: total vs subset DPT (Spearman={corr:.2f})')
    axes[row_i, 1].set_xlabel('total_scanpy_dpt')
    axes[row_i, 1].set_ylabel(spec['pseudotime_col'])

fig.savefig(OUTPUT_DIR / 'subset_recomputed_dpt_diagnostics_pca.png', dpi=220, bbox_inches='tight')
plt.show()

### Heatmaps on Individual Cluster Pseudospace

In [ ]:
subset_dpt_results = {}
subset_pseudotime_columns = {'PT': 'pt_subset_scanpy_dpt', 'TAL': 'tal_subset_scanpy_dpt', 'DCT': 'dct_subset_scanpy_dpt'}
subset_segment_specs = {
    segment: {**spec, 'pseudotime_col': subset_pseudotime_columns[segment]}
    for segment, spec in segment_specs.items()
}

for condition_key, condition_label, condition_mask in [
    ('Mouse_Control', 'Mouse Control', mouse_control_mask),
    ('Mouse_AKI', 'Mouse AKI', mouse_aki_mask),
]:
    for segment, spec in subset_segment_specs.items():
        subset_dpt_results[f'{condition_key}_{segment}'] = plot_marker_heatmap(
            adata_heatmap, spec['markers'], spec['order'], spec['colors'],
        output_dir=OUTPUT_DIR, project_dir=PROJECT_DIR,
            pseudotime_col=spec['pseudotime_col'], cluster_col='broad_tubule_marker_call', cluster_value=segment,
            mask=condition_mask, target_label=condition_label,
            title=f'{condition_label} {segment} markers across subset-recomputed Scanpy DPT',
            output_name=f'{condition_key.lower()}_{segment.lower()}_subset_recomputed_dpt_marker_heatmap.png',
            n_bins=N_BINS, smooth_sigma=SMOOTH_SIGMA, trim_fraction=TRIM_FRACTION, z_clip=spec['z_clip'],
        )

subset_dpt_summary = pd.DataFrame(subset_dpt_results).T
subset_dpt_summary.index.name = 'heatmap'
subset_dpt_summary.to_csv(OUTPUT_DIR / 'subset_recomputed_dpt_heatmap_summary.csv')
display(subset_dpt_summary[['n_cells_before_trim', 'n_cells_trimmed_per_end', 'n_cells', 'cluster_value', 'pseudotime_col', 'dpt_min', 'dpt_max', 'output_path']])

In [ ]:
print('--- Section 3 summary ---')
print(f'Mouse Control tubules: {int(mouse_control_mask.sum()):,}')
print(f'Mouse AKI tubules: {int(mouse_aki_mask.sum()):,}')
print(f'total_scanpy_dpt range: [{adata_heatmap.obs["total_scanpy_dpt"].min():.3f}, {adata_heatmap.obs["total_scanpy_dpt"].max():.3f}]')
for segment in SUBSET_DPT_SPECS:
    col = SUBSET_DPT_SPECS[segment]['pseudotime_col']
    vals = adata_heatmap.obs[col].dropna()
    print(f'{col} range: [{vals.min():.3f}, {vals.max():.3f}] (n={len(vals):,})')
n_pngs = len(list(HEATMAP_OUTPUT_DIR.glob('*.png')))
n_csvs = len(list(HEATMAP_OUTPUT_DIR.glob('*.csv')))
print(f'Wrote {n_pngs} PNGs and {n_csvs} CSVs to {HEATMAP_OUTPUT_DIR.relative_to(PROJECT_DIR)}')

In [ ]:
# Persist the subset-recomputed DPT columns into the DPT file (previously in-memory only).
_subset_cols = [c for c in ['pt_subset_scanpy_dpt', 'tal_subset_scanpy_dpt', 'dct_subset_scanpy_dpt']
                if c in adata_heatmap.obs]
if _subset_cols:
    _dpt = sc.read_h5ad(DPT_OUTPUT_PATH)
    for c in _subset_cols:
        _dpt.obs[c] = adata_heatmap.obs.loc[_dpt.obs_names, c].to_numpy(dtype=float)
    _dpt.write(DPT_OUTPUT_PATH)
    print(f'Wrote subset-DPT columns {_subset_cols} into {DPT_OUTPUT_PATH.name}')

## Section 4 — Healthy vs AKI (mouse-only PT tubules)

Replaces the old notebook-5 GAM approach. Within the cluster-derived **PT** cohort (excluding
`Transitional`), each gene and each pathway module is decomposed along the shared pseudospace
(`total_scanpy_dpt`, already in `[0, 1]`) into three nested penalized-B-spline GAMs on **all** PT
tubules, with condition indicator `c` (`Control`→healthy=0, `IR`→aki=1):

- **M0 (shared):** `intercept + f(s)`
- **M1 (level):** `intercept + f(s) + β·c`
- **M2 (shape):** `intercept + f(s) + β·c + g(s)·c`   (condition×basis interaction)

Smoothing `λ` is **GCV-selected per gene/module** (on M2, reused for M0/M1). Nested F-tests give
`level_F` (M1 vs M0) and `shape_F` (M2 vs M1). Per-condition curves come from the joint M2 model
(prediction at `c=0` vs `c=1`) on the **common-support grid** (intersection of the two conditions'
`[p1, p99]`), so shape is never compared where one condition has no cells. The primary ranking
metric is **`shape_rms`** = RMS over the grid of the offset-removed curve difference
`(curve_aki − curve_healthy) − mean_gap`; `level_effect` = `mean_gap`; `condition_effect_rms`
(total curve difference) is kept for continuity.

**Significance is honest for the 2-vs-2 specimen design:** the healthy/aki label is permuted across
the **4 samples** (all C(4,2)=6 relabelings); `sample_perm_p = (#{perm shape_rms > observed}+1)/6`
(min 1/6). Pseudospace is **never** shuffled within a sample — the old pseudoreplicated
within-sample-shuffle engine (`_gpu_permutation_exceedances` / `empirical_dynamic_pvalues`) is
deliberately dropped. F-statistics are reported as effect-size magnitudes, not powered p-values.

Pathway modules use the shared human-symbol Enrichr libraries (Reactome, Hallmark, KEGG) matched to
mouse `var_names` by case-insensitive symbol collision, against the full tested mouse gene
background; module score = mean per-gene z-scored lognorm (z-scored over the PT subset).

> **Cohort disclaimer.** 2 control + 2 IR specimens. Results are **descriptive and
> effect-size-ranked**, not confirmatory; a formal 2v2 significance test is not achievable
> (permutation p-floor ≈ 1/6), so `sample_perm_p` is a calibration/sanity check only.

All outputs are written to `results/mouse_only/healthy_vs_aki/` (created below). Pathway JSONs in
`data/mouse_vs_human/pathway_gene_sets/` are read-only. Pure Python — no rpy2/tradeSeq/mgcv, no GPU.


In [ ]:
from pseudospace.stats_gam import (
    as_csr,
    zscore_rows,
    gam_internal_knots,
    bh_adjust,
    safe_spearman,
    resolve_present,
)

# ============================================================================
# Section 4.0 -- imports + GAM/stats utilities (copied from notebook 5) + output dir
# ============================================================================
import json
import itertools
import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.stats import spearmanr, wasserstein_distance, ks_2samp
from scipy.ndimage import gaussian_filter1d
from patsy import dmatrix
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
from IPython.display import display, Image

HEALTHY_VS_AKI_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# --- utilities copied verbatim from 5_mouse_healthy_vs_aki.ipynb (CONFIG defaults inlined) ---

In [ ]:
# ----------------------------------------------------------------------------
# Section 4 configuration
# ----------------------------------------------------------------------------
SECTION4_CONFIG = {
    'n_internal_knots': 9,                     # target ~8-10 (vs old default of 3) -> less oversmoothing
    'lambda_grid': np.logspace(-3, 3, 13),     # GCV smoothing search grid (alpha in make_gam_penalty)
    'common_support_pct': (1, 99),             # per-condition [p1, p99]; grid = intersection
    'grid_points': 101,
    'min_detected_fraction': 0.02,             # tested-gene background: detected in >= 2% of PT cells
    'min_mean_expression': 0.0,
    'pathway_min_genes': 10,
    'pathway_max_genes': 150,
    'pathway_libraries': ['Reactome_2022', 'MSigDB_Hallmark_2020', 'KEGG_2019_Mouse'],
    'ranking_metric': 'shape_rms',
    'random_state': RANDOM_STATE,
    # visualization layer
    'N_TOP_PLOT': 12,
    'heatmap_n_bins': N_BINS,                  # reuse NB3 (120)
    'heatmap_trim_fraction': TRIM_FRACTION,    # reuse NB3 (0.05)
    'heatmap_smooth_sigma': SMOOTH_SIGMA,      # reuse NB3 (2.5)
    'diverging_cmap': 'bwr',
    'condition_colors': {'healthy': 'tab:blue', 'aki': 'tab:orange'},
}
MARKER_GROUPS = {'S1': ['Slc5a2'], 'S2': ['Slc22a6', 'Slc34a1', 'Gatm'], 'S3': ['Slc7a13', 'Acsm3']}
for key, val in SECTION4_CONFIG.items():
    print(f'{key:>22}: {val}')

In [ ]:
from pseudospace.levelshape import run_level_shape, sample_perm_pvalues

### 4.1 — PT cohort, shared coordinate, common-support grid, tested genes


In [ ]:
# ----------------------------------------------------------------------------
# Section 4.1 -- PT cohort, shared coordinate, common-support grid, tested genes
# ----------------------------------------------------------------------------
adata_full = sc.read_h5ad(DPT_OUTPUT_PATH)
if 'lognorm' in adata_full.layers:
    adata_full.X = adata_full.layers['lognorm'].copy()

assert not adata_full.obs['broad_tubule_marker_call'].isin(['Glomerulus', 'Vessel']).any(), (
    f'{DPT_OUTPUT_PATH.name} contains Glomerulus/Vessel cells -- this file should only contain '
    'tubule-family + Transitional cells. Re-run Section 2 (which now filters these out before '
    'saving the DPT AnnData) to regenerate it.'
)

# PT subset: cluster-derived broad_tubule_marker_call == 'PT', excluding Transitional.
call = adata_full.obs['broad_tubule_marker_call'].astype(str)
adata_pt = adata_full[call == 'PT'].copy()
assert 'Transitional' not in set(adata_pt.obs['broad_tubule_marker_call'].astype(str)), 'PT subset must exclude Transitional'

# Condition: Control -> healthy (0), IR -> aki (1); matched case-insensitively.
cond_lower = adata_pt.obs['condition'].astype(str).str.lower()
c = (cond_lower == 'ir').astype(float).to_numpy()
samples = adata_pt.obs['sample'].astype(str).to_numpy()
control_samples = sorted(set(samples[c == 0]))
aki_samples = sorted(set(samples[c == 1]))

# Shared pseudospace = total_scanpy_dpt (already [0,1]); do NOT rerank per condition.
s = adata_pt.obs['total_scanpy_dpt'].to_numpy(dtype=float)
assert np.all((s >= 0) & (s <= 1)), 'total_scanpy_dpt must be within [0, 1]'

# Common-support grid = intersection of per-condition [p1, p99].
p_lo, p_hi = SECTION4_CONFIG['common_support_pct']
lo = max(np.percentile(s[c == 0], p_lo), np.percentile(s[c == 1], p_lo))
hi = min(np.percentile(s[c == 0], p_hi), np.percentile(s[c == 1], p_hi))
assert lo < hi, 'empty common support between conditions'
grid = np.linspace(lo, hi, SECTION4_CONFIG['grid_points'])
knots = gam_internal_knots(s, basis_df=3 + SECTION4_CONFIG['n_internal_knots'])

# Tested-gene background: detection filter over PT cells.
Y_all = as_csr(adata_pt.layers['lognorm'])
detected = np.asarray((Y_all > 0).sum(axis=0)).ravel()
gene_mean_all = np.asarray(Y_all.mean(axis=0)).ravel()
min_detected = int(np.ceil(SECTION4_CONFIG['min_detected_fraction'] * adata_pt.n_obs))
tested = (detected >= min_detected) & (gene_mean_all >= SECTION4_CONFIG['min_mean_expression'])
gene_names = adata_pt.var_names.to_numpy()[tested]
gene_local = {g_: i for i, g_ in enumerate(gene_names)}
Y_genes = Y_all[:, tested].tocsr()
Y_csc = Y_genes.tocsc()

# Per-gene mean/std over PT cells (for module z-scores + z-score plotting).
gene_mean = np.asarray(Y_genes.mean(axis=0)).ravel()
gene_sq = np.asarray(Y_genes.multiply(Y_genes).mean(axis=0)).ravel()
gene_std = np.sqrt(np.maximum(gene_sq - gene_mean ** 2, 0.0))
gene_std[gene_std == 0] = 1.0

n_healthy = int((c == 0).sum())
n_aki = int((c == 1).sum())
print(f'PT tubules: {adata_pt.n_obs:,}  (healthy={n_healthy:,}, aki={n_aki:,})')
print(f'Healthy samples: {control_samples}    AKI samples: {aki_samples}')
print(f'Tested genes: {len(gene_names):,} / {adata_pt.n_vars:,}')
print(f'Common support: [{lo:.3f}, {hi:.3f}]   internal knots: {len(knots)}  ({knots.round(3)})')

### 4.3 — Gene-level nested level/shape decomposition + effect sizes


In [ ]:
# ----------------------------------------------------------------------------
# Section 4.3 -- gene-level nested level/shape decomposition + effect sizes
# ----------------------------------------------------------------------------
gene_ls = run_level_shape(Y_genes, s, c, knots, grid, SECTION4_CONFIG['lambda_grid'])
print(f'Gene-level level/shape fit complete for {len(gene_names):,} genes.')
print(f"  shape_rms:    [{gene_ls['shape_rms'].min():.4f}, {gene_ls['shape_rms'].max():.4f}]")
print(f"  |level_eff|:  [{np.abs(gene_ls['level_effect']).min():.4f}, {np.abs(gene_ls['level_effect']).max():.4f}]")

In [ ]:
# ----------------------------------------------------------------------------
# Section 4.4 -- sample-level condition-label permutation (honest for n=2 vs 2)
# ----------------------------------------------------------------------------
gene_shape_p, gene_level_p, splits, true_idx = sample_perm_pvalues(
    Y_genes, s, samples, knots, grid, SECTION4_CONFIG['lambda_grid'],
    gene_ls['lam_idx'], gene_ls['sl2'], gene_ls['p_b'], control_samples)
gene_shape_fdr = bh_adjust(gene_shape_p)

print('=' * 78)
print('SAMPLE-LEVEL PERMUTATION -- CALIBRATION/SANITY CHECK, NOT A POWERED TEST.')
print('With 2 healthy vs 2 AKI specimens there are only C(4,2)=6 sample relabelings;')
print('p = #{splits with permuted shape_rms >= observed}/6 (observed included -> p never 0).')
print('The 6 splits form 3 mirror-image (healthy/aki swap) pairs; shape_rms is symmetric under')
print('swap -> effectively 3 distinct values. RANK BY EFFECT SIZE (shape_rms), not by p.')
print('=' * 78)
print('distinct gene sample_perm_p values:', np.unique(np.round(gene_shape_p, 4)))

### 4.5 — Pathway module scores + level/shape GAM (dense path)


In [ ]:
# ----------------------------------------------------------------------------
# Section 4.5 -- pathway module scores + same level/shape GAM (dense path)
# NB: Reactome/Hallmark/KEGG are human-symbol Enrichr libraries; genes are matched to mouse
# var_names by case-insensitive symbol collision, against the full tested mouse gene background.
# Module score = mean of per-gene z-scored lognorm (z-scored over the PT subset only).
# ----------------------------------------------------------------------------
bg_lookup = {g_.upper(): g_ for g_ in gene_names}
retained = []
for library in SECTION4_CONFIG['pathway_libraries']:
    gene_sets = json.loads((PATHWAY_LIBRARY_DIR / f'{library}.json').read_text())
    for pathway, requested in gene_sets.items():
        present = sorted({bg_lookup[str(x).upper()] for x in requested
                          if isinstance(x, str) and str(x).upper() in bg_lookup})
        if SECTION4_CONFIG['pathway_min_genes'] <= len(present) <= SECTION4_CONFIG['pathway_max_genes']:
            retained.append({'library': library, 'pathway': pathway,
                             'genes_present': present, 'n_genes_present': len(present)})
retained_pathways = pd.DataFrame(retained).reset_index(drop=True)
P = len(retained_pathways)

module_scores = np.empty((adata_pt.n_obs, P), dtype=np.float64)
for j, members in enumerate(retained_pathways['genes_present']):
    idxs = [gene_local[g_] for g_ in members]
    sub = Y_csc[:, idxs].toarray()
    module_scores[:, j] = ((sub - gene_mean[idxs]) / gene_std[idxs]).mean(axis=1)

path_ls = run_level_shape(module_scores, s, c, knots, grid, SECTION4_CONFIG['lambda_grid'])
path_shape_p, path_level_p, _, _ = sample_perm_pvalues(
    module_scores, s, samples, knots, grid, SECTION4_CONFIG['lambda_grid'],
    path_ls['lam_idx'], path_ls['sl2'], path_ls['p_b'], control_samples)
path_shape_fdr = bh_adjust(path_shape_p)
print(f'Pathway modules tested: {P}  '
      f"(shape_rms max={path_ls['shape_rms'].max():.4f})")

### 4.6 — Analysis B: pseudospace-structure diagnostics (secondary)


In [ ]:
# ----------------------------------------------------------------------------
# Section 4.6 -- Analysis B: pseudospace-structure diagnostics (secondary)
# Flags whether a shape hit could be driven by density redistribution rather than expression.
# ----------------------------------------------------------------------------
s_h = s[c == 0]
s_a = s[c == 1]
wass = float(wasserstein_distance(s_h, s_a))
ks_stat, ks_p = ks_2samp(s_h, s_a)


def _marker_axis(group):
    present = resolve_present(MARKER_GROUPS[group], gene_names)
    if not present:
        return np.zeros(adata_pt.n_obs), present
    idxs = [gene_local[g_] for g_ in present]
    sub = Y_csc[:, idxs].toarray()
    return ((sub - gene_mean[idxs]) / gene_std[idxs]).mean(axis=1), present


s1_axis, s1_present = _marker_axis('S1')
s3_axis, s3_present = _marker_axis('S3')
marker_axis = s3_axis - s1_axis                              # low (S1) -> high (S3)
mono_healthy = safe_spearman(marker_axis[c == 0], s[c == 0])
mono_aki = safe_spearman(marker_axis[c == 1], s[c == 1])

structure_diag = pd.DataFrame([
    {'metric': 'wasserstein_distance_healthy_aki', 'value': wass},
    {'metric': 'ks_statistic', 'value': float(ks_stat)},
    {'metric': 'ks_pvalue', 'value': float(ks_p)},
    {'metric': 'marker_axis_monotonicity_spearman_healthy', 'value': mono_healthy},
    {'metric': 'marker_axis_monotonicity_spearman_aki', 'value': mono_aki},
    {'metric': 's1_markers_present', 'value': ';'.join(s1_present)},
    {'metric': 's3_markers_present', 'value': ';'.join(s3_present)},
    {'metric': 'n_cells_healthy', 'value': n_healthy},
    {'metric': 'n_cells_aki', 'value': n_aki},
])
structure_diag.to_csv(HEALTHY_VS_AKI_OUTPUT_DIR / 'pseudospace_structure_diagnostics.csv', index=False)

# per-condition density on the common grid (also reused for the optional overlay)
dens_edges = np.linspace(lo, hi, SECTION4_CONFIG['grid_points'] + 1)
dens_centers = 0.5 * (dens_edges[:-1] + dens_edges[1:])
dens_healthy, _ = np.histogram(s_h, bins=dens_edges, density=True)
dens_aki, _ = np.histogram(s_a, bins=dens_edges, density=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
axes[0].fill_between(dens_centers, dens_healthy, alpha=0.45,
                     color=SECTION4_CONFIG['condition_colors']['healthy'], label='healthy')
axes[0].fill_between(dens_centers, dens_aki, alpha=0.45,
                     color=SECTION4_CONFIG['condition_colors']['aki'], label='aki')
axes[0].set_xlabel('Shared pseudospace')
axes[0].set_ylabel('PT cell density')
axes[0].set_title(f'PT density  (Wasserstein={wass:.3f}, KS p={ks_p:.2g})')
axes[0].legend()
rng = np.random.default_rng(SECTION4_CONFIG['random_state'])
for lab, mask, col in [('healthy', c == 0, SECTION4_CONFIG['condition_colors']['healthy']),
                       ('aki', c == 1, SECTION4_CONFIG['condition_colors']['aki'])]:
    idx = np.flatnonzero(mask)
    sel = rng.choice(idx, size=min(3000, idx.size), replace=False)
    axes[1].scatter(s[sel], marker_axis[sel], s=4, alpha=0.25, color=col, label=lab)
axes[1].set_xlabel('Shared pseudospace')
axes[1].set_ylabel('S1->S3 marker axis  (z(S3) - z(S1))')
axes[1].set_title(f'Marker-axis monotonicity  (healthy rho={mono_healthy:.2f}, aki rho={mono_aki:.2f})')
axes[1].legend()
fig.tight_layout()
fig.savefig(HEALTHY_VS_AKI_OUTPUT_DIR / 'pseudospace_structure_diagnostics.png', bbox_inches='tight', dpi=130)
plt.close(fig)
display(Image(filename=str(HEALTHY_VS_AKI_OUTPUT_DIR / 'pseudospace_structure_diagnostics.png')))
print('Analysis B diagnostics written.')
print(structure_diag.to_string(index=False))

### 4.7 — Result tables


In [ ]:
# ----------------------------------------------------------------------------
# Section 4.7 -- result CSVs (ranked by shape_rms)
# ----------------------------------------------------------------------------
gene_results = pd.DataFrame({
    'gene': gene_names,
    'level_F': gene_ls['level_F'],
    'level_effect': gene_ls['level_effect'],
    'level_abs': np.abs(gene_ls['level_effect']),
    'm2_condition_coef': gene_ls['m2_condition_coef'],
    'shape_F': gene_ls['shape_F'],
    'shape_rms': gene_ls['shape_rms'],
    'condition_effect_rms': gene_ls['condition_effect_rms'],
    'condition_effect_max_abs': gene_ls['condition_effect_max_abs'],
    'curve_spearman': gene_ls['curve_spearman'],
    'amplitude_healthy': gene_ls['amplitude_healthy'],
    'amplitude_aki': gene_ls['amplitude_aki'],
    'sample_perm_p': gene_shape_p,  # FDR omitted: p floored at 1/6 (n=2v2) -> BH non-functional
    'level_perm_p': gene_level_p,
    'n_cells_healthy': n_healthy,
    'n_cells_aki': n_aki,
}).sort_values('shape_rms', ascending=False).reset_index(drop=True)
gene_results.to_csv(HEALTHY_VS_AKI_OUTPUT_DIR / 'gene_level_shape_results.csv', index=False)

pathway_results = retained_pathways.drop(columns='genes_present').assign(
    level_F=path_ls['level_F'],
    level_effect=path_ls['level_effect'],
    level_abs=np.abs(path_ls['level_effect']),
    m2_condition_coef=path_ls['m2_condition_coef'],
    shape_F=path_ls['shape_F'],
    shape_rms=path_ls['shape_rms'],
    condition_effect_rms=path_ls['condition_effect_rms'],
    condition_effect_max_abs=path_ls['condition_effect_max_abs'],
    curve_spearman=path_ls['curve_spearman'],
    sample_perm_p=path_shape_p,
    level_perm_p=path_level_p,
    n_cells_healthy=n_healthy,
    n_cells_aki=n_aki,
).sort_values('shape_rms', ascending=False).reset_index(drop=True)
pathway_results.to_csv(HEALTHY_VS_AKI_OUTPUT_DIR / 'pathway_level_shape_results.csv', index=False)

display(gene_results.head(10))
display(pathway_results[['library', 'pathway', 'n_genes_present', 'level_effect',
                         'shape_rms', 'shape_F', 'sample_perm_p']].head(10))

In [ ]:
# Leave-one-specimen-out robustness of the top shape hits (honest ceiling at n=2 vs 2).
from pseudospace.levelshape import loso_shape_stability
top_idx = np.argsort(gene_ls['shape_rms'])[::-1][:SECTION4_CONFIG['N_TOP_PLOT']]
loso = loso_shape_stability(Y_genes, s, c, samples, knots, grid, SECTION4_CONFIG['lambda_grid'],
                            top_idx, feature_names=gene_names)
loso.to_csv(HEALTHY_VS_AKI_OUTPUT_DIR / 'loso_shape_stability_genes.csv', index=False)
feats = list(gene_names[top_idx])
fig, ax = plt.subplots(figsize=(8, 0.4 * len(feats) + 1.5))
for j, feat in enumerate(feats):
    sub = loso[loso['feature'] == feat]
    ax.plot(sub['loso_shape_rms'], [j] * len(sub), 'o', color='steelblue', alpha=0.7)
    ax.plot([sub['baseline_shape_rms'].iloc[0]], [j], 'D', color='firebrick')
ax.set_yticks(range(len(feats))); ax.set_yticklabels(feats, fontsize=8)
ax.set_xlabel('shape_rms  (diamond = all specimens; circles = leave-one-out)')
ax.set_title('Leave-one-specimen-out robustness of top shape genes')
ax.invert_yaxis()
fig.savefig(HEALTHY_VS_AKI_OUTPUT_DIR / 'loso_shape_stability_genes.png', dpi=130, bbox_inches='tight')
plt.show()
display(loso.head(12))

### 4.8 — Visualization layer (reuses fitted curves + result CSVs; no stat recompute)


In [ ]:
# ----------------------------------------------------------------------------
# Section 4.8 -- VISUALIZATION LAYER (reuses fitted curves + result CSVs; no stat recompute)
# Group 1 -- familiar per-condition trajectory views
# ----------------------------------------------------------------------------
N_TOP = SECTION4_CONFIG['N_TOP_PLOT']
COL_H = SECTION4_CONFIG['condition_colors']['healthy']
COL_A = SECTION4_CONFIG['condition_colors']['aki']
figure_index = []


def _grid_axes(n_panels, ncols=3, panel_h=3.0, panel_w=5.0):
    nrows = max(1, int(np.ceil(n_panels / ncols)))
    fig, axes = plt.subplots(nrows, ncols, figsize=(panel_w * ncols, panel_h * nrows), squeeze=False)
    return fig, axes.ravel()


def _save(fig, name):
    path = HEALTHY_VS_AKI_OUTPUT_DIR / name
    fig.tight_layout()
    fig.savefig(path, bbox_inches='tight', dpi=130)
    plt.close(fig)
    figure_index.append(name)
    display(Image(filename=str(path)))


def plot_trajectory_grid(labels, curve_h, curve_a, ylabel, suptitle, name,
                         densities=None):
    fig, axes = _grid_axes(len(labels))
    for ax, i in zip(axes, range(len(labels))):
        ax.plot(grid, curve_h[i], color=COL_H, lw=2, label='healthy')
        ax.plot(grid, curve_a[i], color=COL_A, lw=2, label='aki')
        if densities is not None:
            twin = ax.twinx()
            twin.fill_between(dens_centers, densities[0], color=COL_H, alpha=0.08)
            twin.fill_between(dens_centers, densities[1], color=COL_A, alpha=0.08)
            twin.set_yticks([])
        ax.set_title(str(labels[i])[:48], fontsize=9)
        ax.set_xlabel('Shared pseudospace')
        ax.set_ylabel(ylabel)
    axes[0].legend(fontsize=8)
    for ax in axes[len(labels):]:
        ax.axis('off')
    fig.suptitle(suptitle, y=1.005, fontsize=12)
    _save(fig, name)


# genes: top-N by shape_rms
gene_top = np.argsort(gene_ls['shape_rms'])[::-1][:N_TOP]
gt_names = gene_names[gene_top]
gt_h_raw = gene_ls['curve_healthy'][gene_top]
gt_a_raw = gene_ls['curve_aki'][gene_top]
gt_mean = gene_mean[gene_top][:, None]
gt_std = gene_std[gene_top][:, None]
gt_h_z = (gt_h_raw - gt_mean) / gt_std
gt_a_z = (gt_a_raw - gt_mean) / gt_std
plot_trajectory_grid(gt_names, gt_h_z, gt_a_z, 'Mean gene z-score',
                     'Top shape-ranked gene trajectories (z-score)',
                     'top_shape_trajectories_zscore_genes.png',
                     densities=(dens_healthy, dens_aki))
plot_trajectory_grid(gt_names, gt_h_raw, gt_a_raw, 'Log-normalized expression',
                     'Top shape-ranked gene trajectories (raw log-normalized)',
                     'top_shape_trajectories_rawlognorm_genes.png')

# pathways: top-N by shape_rms; z-score curves reuse module fit
path_top = np.argsort(path_ls['shape_rms'])[::-1][:N_TOP]
pt_labels = [f"{retained_pathways.loc[i, 'library'].split('_')[0]}: {retained_pathways.loc[i, 'pathway']}"
             for i in path_top]
pt_h_z = path_ls['curve_healthy'][path_top]
pt_a_z = path_ls['curve_aki'][path_top]
plot_trajectory_grid(pt_labels, pt_h_z, pt_a_z, 'Mean gene z-score',
                     'Top shape-ranked pathway trajectories (module z-score)',
                     'top_shape_trajectories_zscore_pathways.png')

# pathways raw log-normalized: refit M2 on raw mean-lognorm module for the top-N (for plotting only)
raw_top = np.empty((adata_pt.n_obs, len(path_top)))
for jj, pi in enumerate(path_top):
    idxs = [gene_local[g_] for g_ in retained_pathways.loc[pi, 'genes_present']]
    raw_top[:, jj] = Y_csc[:, idxs].toarray().mean(axis=1)
raw_fit = run_level_shape(raw_top, s, c, knots, grid, SECTION4_CONFIG['lambda_grid'])
plot_trajectory_grid(pt_labels, raw_fit['curve_healthy'], raw_fit['curve_aki'],
                     'Mean raw log-normalized expression',
                     'Top shape-ranked pathway trajectories (raw log-normalized)',
                     'top_shape_trajectories_rawlognorm_pathways.png')

# canonical S1/S2/S3 marker validation (per condition), reusing gene_ls fitted curves
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharex=True)
for ax, group in zip(axes, ['S1', 'S2', 'S3']):
    for gene in resolve_present(MARKER_GROUPS[group], gene_names):
        gi = gene_local[gene]
        ax.plot(grid, gene_ls['curve_healthy'][gi], color=COL_H, lw=2)
        ax.plot(grid, gene_ls['curve_aki'][gi], color=COL_A, lw=2, ls='--')
        ax.text(grid[-1], gene_ls['curve_healthy'][gi][-1], f' {gene}', fontsize=8, va='center')
    ax.set_title(f'{group} markers')
    ax.set_xlabel('Shared pseudospace')
    ax.set_ylabel('GAM-fitted log-normalized expression')
axes[0].plot([], [], color=COL_H, lw=2, label='healthy')
axes[0].plot([], [], color=COL_A, lw=2, ls='--', label='aki')
axes[0].legend(fontsize=8)
fig.suptitle('Canonical PT S1/S2/S3 marker validation', y=1.02)
_save(fig, 'marker_validation_S1S2S3.png')
print('Group 1 figures written:', len(figure_index))

In [ ]:
# ----------------------------------------------------------------------------
# Group 2 -- level/shape decomposition views (the point of the method)
# ----------------------------------------------------------------------------
def level_shape_scatter(df, name, title):
    fig, ax = plt.subplots(figsize=(7.5, 6))
    x = df['level_effect'].to_numpy()
    y = df['shape_rms'].to_numpy()
    color = -np.log10(np.clip(df['sample_perm_p'].to_numpy(), 1e-6, 1))
    sc_ = ax.scatter(x, y, c=color, cmap='viridis', s=18, alpha=0.7)
    fig.colorbar(sc_, ax=ax, label='-log10(sample_perm_p)')
    y_ref = np.quantile(y, 0.98)
    x_ref = np.quantile(np.abs(x), 0.98)
    ax.axhline(y_ref, color='grey', ls=':', lw=1)
    ax.axvline(x_ref, color='grey', ls=':', lw=1)
    ax.axvline(-x_ref, color='grey', ls=':', lw=1)
    ax.text(0.98, 0.02, 'high |x|, low y = pure level shift (DE-like)\nhigh y = shape change',
            transform=ax.transAxes, ha='right', va='bottom', fontsize=8,
            bbox=dict(boxstyle='round', fc='white', ec='grey', alpha=0.8))
    label_col = 'gene' if 'gene' in df.columns else 'pathway'
    for _, row in df.sort_values('shape_rms', ascending=False).head(N_TOP).iterrows():
        ax.annotate(str(row[label_col])[:26], (row['level_effect'], row['shape_rms']),
                    fontsize=7, xytext=(3, 3), textcoords='offset points')
    ax.set_xlabel('level_effect  (mean healthy->aki vertical gap)')
    ax.set_ylabel('shape_rms  (offset-removed curve difference)')
    ax.set_title(title + '\nDE would surface only the x-axis (level) hits', fontsize=10)
    _save(fig, name)


level_shape_scatter(gene_results, 'level_vs_shape_scatter_genes.png', 'Genes: level vs shape')
level_shape_scatter(pathway_results, 'level_vs_shape_scatter_pathways.png', 'Pathways: level vs shape')


def offset_removed_grid(labels, curve_h, curve_a, name, suptitle):
    fig, axes = _grid_axes(len(labels))
    for ax, i in zip(axes, range(len(labels))):
        diff = curve_a[i] - curve_h[i]
        ax.plot(grid, diff - diff.mean(), color='firebrick', lw=2)
        ax.axhline(0, color='grey', lw=1, ls='--')
        ax.set_title(str(labels[i])[:48], fontsize=9)
        ax.set_xlabel('Shared pseudospace')
        ax.set_ylabel('(aki - healthy) - mean gap')
    for ax in axes[len(labels):]:
        ax.axis('off')
    fig.suptitle(suptitle, y=1.005, fontsize=12)
    _save(fig, name)


offset_removed_grid(gt_names, gt_h_raw, gt_a_raw, 'offset_removed_curves_genes.png',
                    'Offset-removed difference curves (top shape genes)')
offset_removed_grid(pt_labels, pt_h_z, pt_a_z, 'offset_removed_curves_pathways.png',
                    'Offset-removed difference curves (top shape pathways)')


def binned_matrix_profile(mat, s_vals, lo_, hi_, n_bins, sigma):
    """mat: (cells, cols) dense; returns (cols, n_bins) interpolated + smoothed binned means."""
    edges = np.linspace(lo_, hi_, n_bins + 1)
    centers = 0.5 * (edges[:-1] + edges[1:])
    idx = np.clip(np.digitize(s_vals, edges) - 1, 0, n_bins - 1)
    inb = (s_vals >= lo_) & (s_vals <= hi_)
    out = np.full((mat.shape[1], n_bins), np.nan)
    for b in range(n_bins):
        m = inb & (idx == b)
        if m.any():
            out[:, b] = mat[m].mean(axis=0)
    for i in range(out.shape[0]):
        row = out[i]
        finite = np.isfinite(row)
        if not finite.any():
            continue
        if not finite.all():
            row = np.interp(centers, centers[finite], row[finite])
        out[i] = gaussian_filter1d(row, sigma)
    return out, centers


def delta_pseudospace_heatmap(labels, mat_cols, name, suptitle):
    n_bins = SECTION4_CONFIG['heatmap_n_bins']
    sigma = SECTION4_CONFIG['heatmap_smooth_sigma']
    prof_h, centers = binned_matrix_profile(mat_cols[c == 0], s[c == 0], lo, hi, n_bins, sigma)
    prof_a, _ = binned_matrix_profile(mat_cols[c == 1], s[c == 1], lo, hi, n_bins, sigma)
    delta = prof_a - prof_h
    order = np.argsort(np.argmax(np.abs(delta), axis=1))          # order rows by where |delta| peaks
    delta = delta[order]
    labels = [labels[i] for i in order]
    vmax = np.nanpercentile(np.abs(delta), 99) or 1.0
    strip, _ = binned_matrix_profile(marker_axis[:, None], s, lo, hi, n_bins, sigma)
    fig, (ax_s, ax) = plt.subplots(2, 1, figsize=(11, 0.32 * len(labels) + 2.0),
                                   gridspec_kw={'height_ratios': [1, max(len(labels), 6)]},
                                   sharex=True)
    ax_s.imshow(strip, aspect='auto', cmap='PuOr', extent=[lo, hi, 0, 1])
    ax_s.set_yticks([])
    ax_s.set_title('S1 -> S3 marker axis (PuOr)', fontsize=8)
    im = ax.imshow(delta, aspect='auto', cmap=SECTION4_CONFIG['diverging_cmap'],
                   vmin=-vmax, vmax=vmax, extent=[lo, hi, len(labels), 0])
    ax.set_yticks(np.arange(len(labels)) + 0.5)
    ax.set_yticklabels([str(x)[:40] for x in labels], fontsize=7)
    ax.set_xlabel('Shared pseudospace')
    fig.colorbar(im, ax=ax, label='aki - healthy (binned)')
    fig.suptitle(suptitle, y=1.01, fontsize=12)
    _save(fig, name)


delta_pseudospace_heatmap(list(gt_names), Y_csc[:, gene_top].toarray(),
                          'delta_pseudospace_heatmap_genes.png',
                          'Delta (aki - healthy) pseudospace heatmap: top shape genes')
delta_pseudospace_heatmap(list(pt_labels), module_scores[:, path_top],
                          'delta_pseudospace_heatmap_pathways.png',
                          'Delta (aki - healthy) pseudospace heatmap: top shape pathways')


def paired_condition_heatmaps(labels, curve_h, curve_a, name, suptitle):
    zh = zscore_rows(curve_h)
    za = zscore_rows(curve_a)
    fig, axes = plt.subplots(2, 1, figsize=(11, 0.32 * len(labels) + 2.5), sharex=True)
    for ax, mat, title in [(axes[0], zh, 'healthy'), (axes[1], za, 'aki')]:
        im = ax.imshow(mat, aspect='auto', cmap='bwr', vmin=-Z_CLIP, vmax=Z_CLIP,
                       extent=[lo, hi, len(labels), 0])
        ax.set_yticks(np.arange(len(labels)) + 0.5)
        ax.set_yticklabels([str(x)[:40] for x in labels], fontsize=7)
        ax.set_ylabel(title)
        fig.colorbar(im, ax=ax, label='per-row z-score')
    axes[1].set_xlabel('Shared pseudospace')
    fig.suptitle(suptitle, y=1.01, fontsize=12)
    _save(fig, name)


paired_condition_heatmaps(list(gt_names), gt_h_raw, gt_a_raw,
                          'paired_condition_heatmaps_genes.png',
                          'Paired condition heatmaps (top shape genes, shared order)')
paired_condition_heatmaps(list(pt_labels), pt_h_z, pt_a_z,
                          'paired_condition_heatmaps_pathways.png',
                          'Paired condition heatmaps (top shape pathways, shared order)')
print('Group 2 figures written; total so far:', len(figure_index))

In [ ]:
# ----------------------------------------------------------------------------
# Group 3 -- summary / diagnostic views
# ----------------------------------------------------------------------------
def volcano(df, name, title):
    fig, ax = plt.subplots(figsize=(7.5, 6))
    x = df['shape_rms'].to_numpy()
    y = -np.log10(np.clip(df['sample_perm_p'].to_numpy(), 1e-6, 1))
    ax.scatter(x, y, s=16, alpha=0.6, color='slategrey')
    label_col = 'gene' if 'gene' in df.columns else 'pathway'
    for _, row in df.head(N_TOP).iterrows():
        ax.annotate(str(row[label_col])[:24], (row['shape_rms'], -np.log10(max(row['sample_perm_p'], 1e-6))),
                    fontsize=7, xytext=(3, 3), textcoords='offset points')
    ax.set_xlabel('shape_rms (effect size)')
    ax.set_ylabel('-log10(sample_perm_p)')
    ax.text(0.02, 0.98, 'CAVEAT: 2v2 design -> permutation p-floor is coarse (~1/6);\n'
                        'this axis is a calibration check, not a powered test. Rank by shape_rms.',
            transform=ax.transAxes, ha='left', va='top', fontsize=8,
            bbox=dict(boxstyle='round', fc='lightyellow', ec='grey', alpha=0.9))
    ax.set_title(title)
    _save(fig, name)


volcano(gene_results, 'volcano_shape_genes.png', 'Genes: shape effect vs permutation p')
volcano(pathway_results, 'volcano_shape_pathways.png', 'Pathways: shape effect vs permutation p')


def rank_barplot(df, name, title):
    top = df.head(N_TOP).iloc[::-1]
    label_col = 'gene' if 'gene' in df.columns else 'pathway'
    big_level = np.abs(top['level_effect']) >= np.median(np.abs(df['level_effect']))
    colors = ['darkorange' if b else 'steelblue' for b in big_level]
    fig, ax = plt.subplots(figsize=(8, 0.42 * len(top) + 1.5))
    ax.barh([str(x)[:44] for x in top[label_col]], top['shape_rms'], color=colors)
    ax.set_xlabel('shape_rms')
    ax.set_title(title)
    ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color='darkorange'),
                       plt.Rectangle((0, 0), 1, 1, color='steelblue')],
              labels=['also large level shift', 'shape-dominant'], fontsize=8, loc='lower right')
    _save(fig, name)


rank_barplot(gene_results, 'rank_barplot_shape_genes.png', 'Top shape-ranked genes')
rank_barplot(pathway_results, 'rank_barplot_shape_pathways.png', 'Top shape-ranked pathways')

# pathway shape summary matrix: pathway x [level_effect, shape_rms, shape_F, sample_perm_p]
summary_cols = ['level_effect', 'shape_rms', 'shape_F', 'sample_perm_p']
top_paths = pathway_results.head(N_TOP)
mat = top_paths[summary_cols].to_numpy(dtype=float)
norm = np.zeros_like(mat)
for j in range(mat.shape[1]):
    col = mat[:, j]
    rng_ = np.ptp(col)
    norm[:, j] = (col - col.min()) / rng_ if rng_ > 0 else 0.0
fig, ax = plt.subplots(figsize=(7.5, 0.45 * len(top_paths) + 1.8))
im = ax.imshow(norm, aspect='auto', cmap='magma')
ax.set_xticks(range(len(summary_cols)))
ax.set_xticklabels(summary_cols, rotation=30, ha='right')
ax.set_yticks(range(len(top_paths)))
ax.set_yticklabels([str(p)[:44] for p in top_paths['pathway']], fontsize=7)
for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
        ax.text(j, i, f'{mat[i, j]:.2g}', ha='center', va='center', fontsize=7,
                color='white' if norm[i, j] < 0.6 else 'black')
ax.set_title('Top pathway shape summary (color = column min-max normalized)')
_save(fig, 'pathway_shape_summary_matrix.png')
print('Group 3 figures written; total:', len(figure_index))

### 4.9 — Summary


In [ ]:
# ----------------------------------------------------------------------------
# Section 4.9 -- summary + figure index
# ----------------------------------------------------------------------------
print('=' * 78)
print('SECTION 4 SUMMARY -- mouse-only healthy vs AKI (PT tubules)')
print('=' * 78)
print(f'PT tubules: healthy={n_healthy:,}  aki={n_aki:,}  (samples {control_samples} vs {aki_samples})')
print(f'Genes tested: {len(gene_names):,}   Pathway modules tested: {P:,}')
print(f'Common support: [{lo:.3f}, {hi:.3f}]   internal knots: {len(knots)}')
print('\nTop 10 genes by shape_rms:')
print(gene_results.head(10)[['gene', 'level_effect', 'shape_rms', 'shape_F', 'sample_perm_p']].to_string(index=False))
print('\nTop 10 pathways by shape_rms:')
print(pathway_results.head(10)[['library', 'pathway', 'level_effect', 'shape_rms', 'sample_perm_p']].to_string(index=False))
print('\nCOHORT DISCLAIMER: 2 healthy + 2 AKI specimens. Results are DESCRIPTIVE and')
print('EFFECT-SIZE-RANKED (shape_rms). A formal 2v2 significance test is not achievable')
print('(permutation p-floor ~1/6); sample_perm_p is a calibration/sanity check only.')
print('\nFigures written to', str(HEALTHY_VS_AKI_OUTPUT_DIR.relative_to(PROJECT_DIR)) + ':')
for name in figure_index:
    print('  -', name)
csvs = ['gene_level_shape_results.csv', 'pathway_level_shape_results.csv', 'pseudospace_structure_diagnostics.csv']
print('CSVs written:', ', '.join(csvs))

## Section 5 — Three-axis concordance (non-circular fidelity check)

Per specimen, agreement among the DPT pseudospace, the early→late marker axis, and physical
distance-to-nearest-glomerulus (from pass-1 centroids + `coarse_class`). Agreement across three
independent axes is the strongest fidelity evidence achievable at n=2 vs 2.

In [ ]:
from scipy.spatial import cKDTree
from scipy.stats import spearmanr
import scanpy as sc, numpy as np, pandas as pd, matplotlib.pyplot as plt

dpt_ad = sc.read_h5ad(DPT_OUTPUT_PATH)
if 'lognorm' in dpt_ad.layers:
    dpt_ad.X = dpt_ad.layers['lognorm'].copy()
pass1 = sc.read_h5ad(HARMONY_OUTPUT_PATH)   # all cells: coarse_class + x/y centroids

def _score(ad, genes):
    present = [g for g in genes if g in ad.var_names]
    sc.tl.score_genes(ad, present, score_name='_tmp', use_raw=False)
    v = ad.obs['_tmp'].to_numpy(dtype=float); del ad.obs['_tmp']; return v
dpt_ad.obs['total_marker_axis'] = (_score(dpt_ad, TOTAL_POSITION_MARKERS['late'])
                                   - _score(dpt_ad, TOTAL_POSITION_MARKERS['early']))

def _sp(a, b):
    ok = np.isfinite(a) & np.isfinite(b)
    return float(spearmanr(a[ok], b[ok]).correlation) if ok.sum() > 10 else np.nan

rows = []
for sample in sorted(dpt_ad.obs['sample'].astype(str).unique()):
    m = (dpt_ad.obs['sample'].astype(str) == sample).to_numpy()
    obs = dpt_ad.obs[m]
    p1 = pass1.obs[pass1.obs['sample'].astype(str) == sample]
    glom = p1[p1['coarse_class'].astype(str) == 'Glomerulus']
    if len(glom) < 3:
        print(f'{sample}: <3 glomeruli, skipping'); continue
    tree = cKDTree(glom[['x_centroid', 'y_centroid']].to_numpy())
    xy = pass1.obs.loc[obs.index, ['x_centroid', 'y_centroid']].to_numpy()
    dist, _ = tree.query(xy, k=1)
    dpt = obs['total_scanpy_dpt'].to_numpy(dtype=float)
    ma = obs['total_marker_axis'].to_numpy(dtype=float)
    rows.append({'sample': sample, 'n': int(m.sum()),
                 'rho_dpt_vs_dist_glom': _sp(dpt, dist),
                 'rho_marker_vs_dist_glom': _sp(ma, dist),
                 'rho_dpt_vs_marker': _sp(dpt, ma)})
conc = pd.DataFrame(rows)
conc.to_csv(CONCORDANCE_DIR / 'three_axis_concordance.csv', index=False)
display(conc)
fig, ax = plt.subplots(figsize=(7, 4))
conc.set_index('sample')[['rho_dpt_vs_dist_glom', 'rho_marker_vs_dist_glom', 'rho_dpt_vs_marker']].plot(kind='bar', ax=ax)
ax.axhline(0, color='k', lw=0.8); ax.set_ylabel('Spearman rho')
ax.set_title('Three-axis concordance per specimen')
fig.savefig(CONCORDANCE_DIR / 'three_axis_concordance.png', dpi=130, bbox_inches='tight'); plt.show()